<a href="https://colab.research.google.com/github/Parameshwar93/LLM_From_Scratch/blob/main/multimodal_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Simple Tokenizer and Embedding Layer Implementation

We will implement:
1. A simple word-level **Tokenizer** that maps words to unique integer tokens.
2. An **Embedding Layer** (using PyTorch) that maps these tokens to continuous vector spaces.

In [25]:
import torch
import torch.nn as nn
import re

class SimpleTokenizer:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        # Reserve special tokens
        self.vocab['<PAD>'] = 0
        self.vocab['<UNK>'] = 1
        self.inverse_vocab[0] = '<PAD>'
        self.inverse_vocab[1] = '<UNK>'

    def fit(self, texts):
        """Build vocabulary from list of texts"""
        unique_words = set()
        for text in texts:
            words = self._tokenize(text)
            unique_words.update(words)

        for word in sorted(unique_words):
            if word not in self.vocab:
                idx = len(self.vocab)
                self.vocab[word] = idx
                self.inverse_vocab[idx] = word

    def _tokenize(self, text):
        """Basic lowercase word tokenization"""
        return re.findall(r'\w+', text.lower())

    def encode(self, text, max_length=None):
        """Convert text to sequence of token IDs with padding"""
        words = self._tokenize(text)
        tokens = [self.vocab.get(word, self.vocab['<UNK>']) for word in words]

        if max_length is not None:
            if len(tokens) < max_length:
                tokens = tokens + [self.vocab['<PAD>']] * (max_length - len(tokens))
            else:
                tokens = tokens[:max_length]
        return tokens

    def decode(self, tokens):
        """Convert list of token IDs back to text"""
        return ' '.join([self.inverse_vocab.get(t, '<UNK>') for t in tokens if t != self.vocab['<PAD>']])

In [26]:
# Sample Corpus
corpus = [
    "Deep learning is fascinating.",
    "Natural Language Processing and embeddings make it powerful.",
    "We love building neural networks!"
]

# Initialize and fit tokenizer
tokenizer = SimpleTokenizer()
tokenizer.fit(corpus)

print("Vocabulary Size:", len(tokenizer.vocab))
print("Vocabulary Map:", tokenizer.vocab)

# Encode sample sentences
max_len = 8
sample_sentence = "Deep learning makes embeddings powerful."
encoded_sequence = tokenizer.encode(sample_sentence, max_length=max_len)

print(f"\nOriginal: '{sample_sentence}'")
print(f"Encoded (Padded to {max_len}): {encoded_sequence}")
print(f"Decoded: {tokenizer.decode(encoded_sequence)}")

Vocabulary Size: 19
Vocabulary Map: {'<PAD>': 0, '<UNK>': 1, 'and': 2, 'building': 3, 'deep': 4, 'embeddings': 5, 'fascinating': 6, 'is': 7, 'it': 8, 'language': 9, 'learning': 10, 'love': 11, 'make': 12, 'natural': 13, 'networks': 14, 'neural': 15, 'powerful': 16, 'processing': 17, 'we': 18}

Original: 'Deep learning makes embeddings powerful.'
Encoded (Padded to 8): [4, 10, 1, 5, 16, 0, 0, 0]
Decoded: deep learning <UNK> embeddings powerful


In [29]:
# Define PyTorch Embedding Layer
vocab_size = len(tokenizer.vocab)
embedding_dim = 4  # Dimensions of each dense vector representation

# nn.Embedding maps sequence indices to continuous space of shape (vocab_size, embedding_dim)
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

# Convert tokenized sequence into PyTorch Tensor
# shape: [batch_size=1, sequence_length=8]
input_tensor = torch.tensor([encoded_sequence], dtype=torch.long)

# Run forward pass
embedded_vectors = embedding_layer(input_tensor)

print("Input Tensor shape:", input_tensor.shape)
print("Embedding Output shape:", embedded_vectors.shape) # Shape: [batch, seq_len, embedding_dim]
print("\nEmbedding vectors for each token in our input sentence:")
for word, vector in zip(tokenizer.decode(encoded_sequence).split(), embedded_vectors[0]):
    print(f"{word:<12} -> {vector.detach().numpy()}")

Input Tensor shape: torch.Size([1, 8])
Embedding Output shape: torch.Size([1, 8, 4])

Embedding vectors for each token in our input sentence:
deep         -> [-0.7921284  -0.35560858  0.25280693  0.74070686]
learning     -> [0.06449926 0.35754022 1.0277957  0.3703544 ]
<UNK>        -> [-1.7582825   1.1797777   0.34592938 -0.16999826]
embeddings   -> [ 0.600694  -1.7020842  1.3402088  1.1321037]
powerful     -> [-1.1383296  -0.6812623  -0.03119779  1.2473494 ]


### 2. Single Neuron and Backpropagation Implementation

We will implement:
1. A single neuron (perceptron) with a **Sigmoid** activation function.
2. **Forward propagation** to calculate the output and mean squared error (MSE) loss.
3. **Backward propagation** to compute the gradients of the loss with respect to weights and bias using the chain rule.
4. **Gradient descent** to update the weights.

In [30]:
import numpy as np

class SingleNeuron:
    def __init__(self, num_inputs, learning_rate=0.1):
        # Initialize weights and bias randomly
        self.weights = np.random.randn(num_inputs)
        self.bias = np.random.randn()
        self.lr = learning_rate

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _sigmoid_derivative(self, a):
        # Sigmoid derivative with respect to its input z, expressed using its activation output 'a'
        return a * (1 - a)

    def forward(self, x):
        self.x = np.array(x)
        # Linear combination: z = w · x + b
        self.z = np.dot(self.weights, self.x) + self.bias
        # Activation: a = sigmoid(z)
        self.a = self._sigmoid(self.z)
        return self.a

    def backward(self, y_true):
        # Loss = 0.5 * (a - y_true)^2
        # dLoss/da = a - y_true
        da = self.a - y_true

        # da/dz = sigmoid'(z) = a * (1 - a)
        dz = da * self._sigmoid_derivative(self.a)

        # dz/dw = x  => dLoss/dw = (dLoss/da) * (da/dz) * (dz/dw) = dz * x
        d_weights = dz * self.x

        # dz/db = 1  => dLoss/db = dz
        d_bias = dz

        # Update weights and bias using gradient descent
        self.weights -= self.lr * d_weights
        self.bias -= self.lr * d_bias

        # Return loss for tracking
        loss = 0.5 * (self.a - y_true) ** 2
        return loss

In [31]:
# Training data for a simple logical AND gate behavior with 2 inputs
# Inputs: X, Target: Y
training_data = [
    ([0, 0], 0),
    ([0, 1], 0),
    ([1, 0], 0),
    ([1, 1], 1)
]

# Initialize neuron
neuron = SingleNeuron(num_inputs=2, learning_rate=0.2)

print("--- Training Progress ---")
epochs = 100
for epoch in range(epochs):
    epoch_loss = 0
    for x, y_true in training_data:
        # Forward pass
        y_pred = neuron.forward(x)
        # Backward pass and weight updates
        loss = neuron.backward(y_true)
        epoch_loss += loss

    # Print loss every 200 epochs
    if epoch % 200 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:03d} | Total Epoch Loss: {epoch_loss:.6f}")

print("\n--- Trained Model Performance ---")
print(f"Final Weights: {neuron.weights}")
print(f"Final Bias: {neuron.bias:.4f}")
for x, y_true in training_data:
    output = neuron.forward(x)
    print(f"Input: {x} -> Predicted: {output:.4f} (Target: {y_true})")

--- Training Progress ---
Epoch 000 | Total Epoch Loss: 0.819235
Epoch 099 | Total Epoch Loss: 0.292770

--- Trained Model Performance ---
Final Weights: [0.92649874 0.24640217]
Final Bias: -1.2501
Input: [0, 0] -> Predicted: 0.2227 (Target: 0)
Input: [0, 1] -> Predicted: 0.2682 (Target: 0)
Input: [1, 0] -> Predicted: 0.4198 (Target: 0)
Input: [1, 1] -> Predicted: 0.4807 (Target: 1)


### 3. Self-Attention using Matrix Multiplication

Self-attention allows tokens in a sequence to look at other tokens to capture context.
Using Query ($Q$), Key ($K$), and Value ($V$) matrices, the formula for **Scaled Dot-Product Attention** is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

where $d_k$ is the dimension of the key vectors.

In [32]:
import torch
import torch.nn.functional as F

class SelfAttention(torch.nn.Module):
    def __init__(self, embed_dim):
        super(SelfAttention, self).__init__()
        self.embed_dim = embed_dim

        # Linear projections for Query, Key, and Value
        self.q_proj = torch.nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = torch.nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = torch.nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        # x shape: [batch_size, sequence_length, embed_dim]
        batch_size, seq_len, embed_dim = x.shape

        # 1. Project inputs to Q, K, V matrices
        Q = self.q_proj(x)  # [batch_size, seq_len, embed_dim]
        K = self.k_proj(x)  # [batch_size, seq_len, embed_dim]
        V = self.v_proj(x)  # [batch_size, seq_len, embed_dim]

        # 2. Calculate attention scores: Q * K^T
        # We transpose the last two dimensions of K for matrix multiplication
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # 3. Scale the scores
        d_k = embed_dim
        scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 4. Apply Softmax to get attention weights
        attention_weights = F.softmax(scaled_scores, dim=-1)

        # 5. Multiply weights by V to get the context-aware output
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [33]:
# Initialize self-attention layer
embed_dim = 4
attention_layer = SelfAttention(embed_dim=embed_dim)

# Simulate a batch of 1 sequence with 3 words (each has embed_dim=4)
# Shape: [batch_size, seq_length, embed_dim]
sample_input = torch.randn(1, 3, embed_dim)

output, weights = attention_layer(sample_input)

print("Input tensor:\n", sample_input)
print("\nAttention Weights (softmax distributions per token):\n", weights.detach().numpy())
print("\nOutput tensor (contextualized embeddings):\n", output.detach().numpy())

Input tensor:
 tensor([[[ 1.9034, -0.4291,  0.5477, -1.9614],
         [-0.4076, -2.0357, -1.5982, -0.4645],
         [-1.3224,  1.4116, -2.0985,  0.6281]]])

Attention Weights (softmax distributions per token):
 [[[0.6888903  0.22447333 0.08663636]
  [0.16463937 0.17830797 0.65705264]
  [0.11326274 0.16294639 0.7237909 ]]]

Output tensor (contextualized embeddings):
 [[[-0.9722878  -0.8687541   0.53326356  0.5449389 ]
  [ 0.6916269   0.95342314  0.5418714   0.34809607]
  [ 0.8582153   1.1500691   0.54406273  0.33240694]]]


### 4. Multi-Head Causal Attention

In autoregressive decoder models (like GPT), we need **Multi-Head Causal Attention**:
1. **Multi-Head Attention**: Allows the model to jointly attend to information from different representation subspaces at different positions.
2. **Causal Masking**: Ensures that the representation for a given token only depends on previous tokens (preventing looking into the future) by masking upper-triangular positions of the attention matrix with $-\infty$ before applying the softmax.

In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by the number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Project inputs to Q, K, V
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        # Project to Q, K, V and reshape to [batch, seq_len, num_heads, head_dim]
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Transpose to [batch, num_heads, seq_len, head_dim] for batched matrix multiplication
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Compute scaled raw dot-product scores: [batch, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Create an upper triangular causal mask
        # 1s on diagonal and below, 0s above diagonal
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).view(1, 1, seq_len, seq_len)

        # Fill upper triangular part with large negative value
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Compute contextual values and transpose back to [batch, seq_len, num_heads, head_dim]
        out = torch.matmul(attention_weights, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)

        # Final linear projection
        return self.out_proj(out), attention_weights

In [35]:
# Initialize causal multi-head attention
embed_dim = 8
num_heads = 2
causal_attention = MultiHeadCausalAttention(embed_dim=embed_dim, num_heads=num_heads)

# Input sequence representing 4 tokens (e.g., "the", "cat", "sat", "on")
# Shape: [batch_size=1, seq_len=4, embed_dim=8]
sample_seq = torch.randn(1, 4, embed_dim)

output, attention_maps = causal_attention(sample_seq)

print("Input Shape:", sample_seq.shape)
print("Output Shape:", output.shape)
print("\nAttention Map for Head 0 (Lower-triangular causal structure):\n")
# Let's inspect the weights for the first batch, first head
import pandas as pd
attention_matrix_head0 = attention_maps[0, 0].detach().numpy()
pd.DataFrame(attention_matrix_head0,
             index=[f"Token {i}" for i in range(4)],
             columns=[f"Token {j}" for j in range(4)])

Input Shape: torch.Size([1, 4, 8])
Output Shape: torch.Size([1, 4, 8])

Attention Map for Head 0 (Lower-triangular causal structure):



,Token 0,Token 1,Token 2,Token 3
Token 0,1.000000,0.000000,0.000000,0.000000
Token 1,0.445974,0.554026,0.000000,0.000000
Token 2,0.455585,0.385261,0.159154,0.000000
Token 3,0.377856,0.370455,0.138267,0.113423


### 5. Advanced LLM Components: RMSNorm, RoPE, and SwiGLU

Modern architectures like LLaMA replace traditional LayerNorm, sinusoidal/learned absolute positional encodings, and GeLU/ReLU with more efficient and effective components:
1. **RMSNorm (Root Mean Square Normalization)**: Normalizes activations by their root mean square rather than variance and mean, saving computational cost.
2. **RoPE (Rotary Position Embedding)**: Applies a rotation matrix to the 2D slices of Query and Key representations to encode relative positions.
3. **SwiGLU (Swish Gated Linear Unit)**: A GLU variant utilizing the Swish (SiLU) activation function.

In [36]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale parameter (gamma)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: [batch_size, seq_len, dim]
        variance = x.pow(2).mean(-1, keepdim=True)
        # Normalize and scale
        return x * torch.rsqrt(variance + self.eps) * self.weight

In [37]:
class RoPE(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        # dim should be head_dim
        self.dim = dim

        # Compute the frequencies (inv_freq)
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Precompute frequencies over max sequence length
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqss = torch.outer(t, self.inv_freq)
        # Create a polar coordinate representation: [max_seq_len, dim // 2]
        # We duplicate it to match the full head dimension
        emb = torch.cat((freqss, freqss), dim=-1)

        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def _rotate_half(self, x):
        # Helper to rotate 2D vectors by 90 degrees: [-x2, x1]
        x1 = x[..., :self.dim // 2]
        x2 = x[..., self.dim // 2:]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x, seq_len):
        # x shape: [batch_size, num_heads, seq_len, head_dim]
        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(1) # [1, 1, seq_len, head_dim]
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(1)

        # Apply rotation: x * cos(m*theta) + rotate_half(x) * sin(m*theta)
        return (x * cos) + (self._rotate_half(x) * sin)

In [38]:
class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        # SwiGLU requires two parallel linear projections: W_gate and W_candidate
        self.w_gate = nn.Linear(in_features, hidden_features, bias=False)
        self.w_candidate = nn.Linear(in_features, hidden_features, bias=False)
        self.w_out = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        # SwiGLU(x) = (Swish(xW_gate) * xW_candidate) * W_out
        gate = torch.sigmoid(self.w_gate(x)) * self.w_gate(x) # SiLU / Swish activation
        candidate = self.w_candidate(x)
        return self.w_out(gate * candidate)

In [39]:
# Demonstration / Verification
print("--- 1. Testing RMSNorm ---")
rmsnorm = RMSNorm(dim=8)
x_sample = torch.randn(2, 3, 8)
out_rmsnorm = rmsnorm(x_sample)
print("Input shape:", x_sample.shape)
print("Output shape:", out_rmsnorm.shape)
print("RMS of output features (should be close to 1.0):", torch.sqrt(out_rmsnorm.pow(2).mean(-1)))

print("\n--- 2. Testing RoPE ---")
head_dim = 8
rope = RoPE(dim=head_dim)
q_vectors = torch.randn(1, 2, 4, head_dim) # [batch, heads, seq_len, head_dim]
q_rotated = rope(q_vectors, seq_len=4)
print("Rotated shape:", q_rotated.shape)

print("\n--- 3. Testing SwiGLU ---")
swiglu = SwiGLU(in_features=8, hidden_features=16)
out_swiglu = swiglu(x_sample)
print("SwiGLU output shape:", out_swiglu.shape)

--- 1. Testing RMSNorm ---
Input shape: torch.Size([2, 3, 8])
Output shape: torch.Size([2, 3, 8])
RMS of output features (should be close to 1.0): tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]], grad_fn=<SqrtBackward0>)

--- 2. Testing RoPE ---
Rotated shape: torch.Size([1, 2, 4, 8])

--- 3. Testing SwiGLU ---
SwiGLU output shape: torch.Size([2, 3, 8])


### 6. Assembling a Complete GPT-style Transformer

Now, we will combine our components into a complete **Decoder Block** (or Transformer Block) and assemble a mini **GPT-style model** consisting of:
1. An embedding layer (token + position embeddings).
2. Stacked Transformer Blocks (with causal attention, RMSNorm, and SwiGLU FFN).
3. A language modeling projection head.

In [40]:
class GPTTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_features):
        super().__init__()
        self.ln1 = RMSNorm(embed_dim)
        self.attn = MultiHeadCausalAttention(embed_dim, num_heads)
        self.ln2 = RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, hidden_features)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        # Pre-LN residual stream architecture
        attn_out, weights = self.attn(self.ln1(x))
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, weights

In [41]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_features, num_layers, max_seq_len):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_seq_len, embed_dim)

        # Stack of Transformer Decoder Blocks
        self.blocks = nn.ModuleList([
            GPTTransformerBlock(embed_dim, num_heads, hidden_features)
            for _ in range(num_layers)
        ])

        self.ln_f = RMSNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx):
        batch_size, seq_len = idx.shape
        positions = torch.arange(0, seq_len, device=idx.device).unsqueeze(0) # [1, seq_len]

        # Combine token and spatial positional embeddings
        x = self.token_embeddings(idx) + self.position_embeddings(positions)

        all_attention_weights = []
        for block in self.blocks:
            x, attn_w = block(x)
            all_attention_weights.append(attn_w)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, all_attention_weights

In [42]:
# Initialize our mini GPT model
vocab_size = len(tokenizer.vocab)
gpt_model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=8
)

# Prepare input using our earlier tokenizer
test_sentence = "Deep learning make embeddings powerful."
input_tokens = torch.tensor([tokenizer.encode(test_sentence, max_length=8)], dtype=torch.long)

# Forward pass
logits, attention_weights = gpt_model(input_tokens)

print("Input Tokens:", input_tokens)
print("Logits Shape (batch, seq_len, vocab_size):", logits.shape)
print("Number of stacked blocks:", len(attention_weights))

Input Tokens: tensor([[ 4, 10, 12,  5, 16,  0,  0,  0]])
Logits Shape (batch, seq_len, vocab_size): torch.Size([1, 8, 19])
Number of stacked blocks: 2


### 7. Pretraining the Miniature Language Model

To pretrain our model, we will:
1. Prepare training sequences where the inputs ($X$) are shifted from the targets ($Y$) by one token (for next-token prediction).
2. Set up a basic PyTorch training loop using the `CrossEntropyLoss` and the `Adam` optimizer.
3. Track the loss as the model learns to associate context words.

In [43]:
import torch.optim as optim

# 1. Prepare training data
# We will use our corpus and break it down into sequences of length 7 as input, and the shifted target of length 7.
max_seq_len = 8

X_train = []
Y_train = []

for text in corpus:
    # Encode to maximum length
    encoded = tokenizer.encode(text, max_length=max_seq_len)
    # Input: first (max_seq_len - 1) tokens
    X_train.append(encoded[:-1])
    # Target: tokens shifted by one
    Y_train.append(encoded[1:])

X_train = torch.tensor(X_train, dtype=torch.long) # [batch_size, seq_len-1]
Y_train = torch.tensor(Y_train, dtype=torch.long) # [batch_size, seq_len-1]

# 2. Re-initialize a fresh model for training
embed_dim = 16
model = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=embed_dim,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=max_seq_len
)

# 3. Training Configurations
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <PAD> tokens in loss calculation
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. Training loop
model.train()
print("--- Beginning Mini-GPT Pretraining ---")
for epoch in range(150):
    optimizer.zero_grad()

    # Forward pass: outputs are [batch_size, seq_len-1, vocab_size]
    logits, _ = model(X_train)

    # Reshape for loss calculation
    loss = criterion(logits.view(-1, vocab_size), Y_train.view(-1))

    # Backward pass and optimization step
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:03d} / 150 | CrossEntropy Loss: {loss.item():.4f}")

--- Beginning Mini-GPT Pretraining ---
Epoch 025 / 150 | CrossEntropy Loss: 0.0972
Epoch 050 / 150 | CrossEntropy Loss: 0.0121
Epoch 075 / 150 | CrossEntropy Loss: 0.0062
Epoch 100 / 150 | CrossEntropy Loss: 0.0044
Epoch 125 / 150 | CrossEntropy Loss: 0.0033
Epoch 150 / 150 | CrossEntropy Loss: 0.0026


In [44]:
# Test inference of the pretrained model
model.eval()
with torch.no_grad():
    # Start sequence with "Deep learning"
    prompt = "Deep learning"
    tokens = tokenizer.encode(prompt, max_length=7)[:-1]
    test_input = torch.tensor([tokens], dtype=torch.long)

    logits, _ = model(test_input)
    predicted_ids = torch.argmax(logits, dim=-1)[0].tolist()

    print(f"Prompt: '{prompt}'")
    print("Model predictions for next tokens:", tokenizer.decode(predicted_ids))

Prompt: 'Deep learning'
Model predictions for next tokens: learning is networks networks make it


### 8. Autoregressive Inference and Key-Value (KV) Caching

When generating text autoregressively (one token at a time), recalculating the keys ($K$) and values ($V$) for all historical tokens at every single step is highly inefficient ($O(N^2)$ extra operations).

**KV Caching** stores the key and value representations of past tokens in memory, so that at step $t$, we only compute $Q$, $K$, and $V$ for the newly appended token $t$, concatenate the new $K$ and $V$ with the cache, and calculate attention.

In [45]:
class KVCachedMultiHeadCausalAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, kv_cache=None):
        # x shape: [batch_size, seq_len, embed_dim]
        # During incremental generation, seq_len is typically 1
        batch_size, seq_len, embed_dim = x.shape

        # Project to Q, K, V and split heads
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # If KV cache exists, concatenate along the sequence dimension (dim=-2)
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            K = torch.cat([k_prev, K], dim=-2)
            V = torch.cat([v_prev, V], dim=-2)

        new_kv_cache = (K, V)
        total_seq_len = K.shape[-2]

        # Compute scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Causal mask based on the current step's position relative to all historical positions
        # The query vector at step t can only attend to keys up to index t
        mask = torch.ones((seq_len, total_seq_len), device=x.device)
        mask = torch.tril(mask, diagonal=total_seq_len - seq_len).view(1, 1, seq_len, total_seq_len)
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attention_weights, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        return self.out_proj(out), new_kv_cache

In [46]:
class KVCachedGPTTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_features):
        super().__init__()
        self.ln1 = RMSNorm(embed_dim)
        self.attn = KVCachedMultiHeadCausalAttention(embed_dim, num_heads)
        self.ln2 = RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, hidden_features)

    def forward(self, x, kv_cache=None):
        normalized_x = self.ln1(x)
        attn_out, new_kv_cache = self.attn(normalized_x, kv_cache=kv_cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_kv_cache

In [47]:
class CachedMiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_features, num_layers, max_seq_len):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_seq_len, embed_dim)

        self.blocks = nn.ModuleList([
            KVCachedGPTTransformerBlock(embed_dim, num_heads, hidden_features)
            for _ in range(num_layers)
        ])
        self.ln_f = RMSNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx, kv_caches=None, start_pos=0):
        # idx shape: [batch_size, seq_len]
        batch_size, seq_len = idx.shape

        # Position indices for the current slice of tokens
        positions = torch.arange(start_pos, start_pos + seq_len, device=idx.device).unsqueeze(0)

        x = self.token_embeddings(idx) + self.position_embeddings(positions)

        new_kv_caches = []
        for i, block in enumerate(self.blocks):
            block_cache = kv_caches[i] if kv_caches is not None else None
            x, new_cache = block(x, kv_cache=block_cache)
            new_kv_caches.append(new_cache)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, new_kv_caches

In [48]:
def generate_autoregressive(model, start_tokens, max_new_tokens):
    model.eval()
    # Convert prompt tokens to tensor
    idx = torch.tensor([start_tokens], dtype=torch.long)

    # Phase 1: Prefill phase (compute initial KV Cache for the whole prompt)
    logits, kv_caches = model(idx)

    # Pick the prediction for the last token in the prompt sequence
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = list(start_tokens) + [next_token.item()]

    # Phase 2: Incremental decoding using KV Caching
    start_pos = idx.shape[1]
    for _ in range(max_new_tokens - 1):
        # We only pass the single last-generated token
        logits, kv_caches = model(next_token, kv_caches=kv_caches, start_pos=start_pos)
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated.append(next_token.item())
        start_pos += 1

    return generated

# Initialize Cached GPT and load state dict from our pretrained model
cached_gpt = CachedMiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=8
)
cached_gpt.load_state_dict(model.state_dict())

# Test Autoregressive Generation
prompt = "Natural Language"
prompt_tokens = tokenizer.encode(prompt, max_length=5)[:-1]  # Exclude some padding
generated_token_ids = generate_autoregressive(cached_gpt, prompt_tokens, max_new_tokens=4)

print(f"Prompt: '{prompt}'")
print("Autoregressively Generated Tokens:", generated_token_ids)
print("Decoded Result:", tokenizer.decode(generated_token_ids))

Prompt: 'Natural Language'
Autoregressively Generated Tokens: [13, 9, 0, 0, 14, 12, 8, 16]
Decoded Result: natural language networks make it powerful


### 9. LoRA (Low-Rank Adaptation) Implementation from Scratch

Instead of updating the entire weight matrix $W_0 \in \mathbb{R}^{d \times k}$ during fine-tuning, **LoRA** represents the weight update $\Delta W$ using a low-rank decomposition:

$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \times A)$$

where:
* $W_0$ is frozen.
* $A \in \mathbb{R}^{r \times k}$ is initialized from a Gaussian distribution.
* $B \in \mathbb{R}^{d \times r}$ is initialized to 0 (ensuring $\Delta W = 0$ at the start of training).
* $r$ is the rank (typically $r \ll \min(d, k)$).
* $\alpha$ is a constant scaling hyperparameter.

In [49]:
import math
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, r: int = 4, alpha: int = 8):
        super().__init__()
        self.original_linear = original_linear
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze original linear layer weights
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False

        # Define trainable low-rank adaptation matrices
        in_features = original_linear.in_features
        out_features = original_linear.out_features

        # B: scales to output dimension, initialized to 0
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        # A: compresses input dimension, initialized with Gaussian
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))

        # Reset parameters
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        # Standard projection output: xW_0^T
        base_output = self.original_linear(x)

        # Low-rank adaptation: x(B A)^T = x A^T B^T
        # Equivalent to applying A then B directly to the input:
        lora_output = (x @ self.lora_A.t()) @ self.lora_B.t()

        # Combine base and scaled low-rank update
        return base_output + lora_output * self.scaling

In [50]:
# 1. Create a dummy base linear layer representing a pretrained projection layer
original_projection = nn.Linear(in_features=8, out_features=8)

# 2. Wrap it with our LoRA layer (using low rank r=2)
lora_projection = LoRALinear(original_projection, r=2, alpha=4)

# 3. Print out trainable parameters to verify freezing works
print("--- Trainable Parameters (Only LoRA weights should be trainable) ---")
for name, param in lora_projection.named_parameters():
    print(f"{name:<45} | Requires Grad: {param.requires_grad} | Shape: {list(param.shape)}")

# 4. Forward pass verification
sample_input = torch.randn(1, 3, 8)
output = lora_projection(sample_input)
print(f"\nInput Shape: {sample_input.shape}")
print(f"Output Shape: {output.shape}")

--- Trainable Parameters (Only LoRA weights should be trainable) ---
lora_B                                        | Requires Grad: True | Shape: [8, 2]
lora_A                                        | Requires Grad: True | Shape: [2, 8]
original_linear.weight                        | Requires Grad: False | Shape: [8, 8]
original_linear.bias                          | Requires Grad: False | Shape: [8]

Input Shape: torch.Size([1, 3, 8])
Output Shape: torch.Size([1, 3, 8])


### 10. Sparse Mixture of Experts (MoE) Layer

A **Mixture of Experts (MoE)** replaces dense FFN layers with a collection of specialized networks ("experts") and a routing/gating network. For each token, the gating network calculates routing probabilities and forwards the token only to the top-$k$ selected experts. This keeps the active compute constant while scaling the model's capacity.

In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """A single Feed-Forward Network acting as an expert"""
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_experts=4, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Define the set of experts
        self.experts = nn.ModuleList([Expert(embed_dim, hidden_dim) for _ in range(num_experts)])

        # Gating network calculates a probability distribution over the experts
        self.gate = nn.Linear(embed_dim, num_experts, bias=False)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        orig_shape = x.shape
        x_flat = x.view(-1, orig_shape[-1])  # Flatten to [tokens, embed_dim]
        num_tokens = x_flat.size(0)

        # 1. Compute routing logits
        gate_logits = self.gate(x_flat) # [num_tokens, num_experts]

        # 2. Select top-k experts
        topk_weights, topk_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        topk_weights = F.softmax(topk_weights, dim=-1) # Normalize weights among chosen experts

        # 3. Route tokens to respective experts and combine their outputs
        out_flat = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            # Find which tokens are routed to expert i
            # token_indices holds index of the token, and k_idx indicates whether it was 1st or 2nd choice
            token_indices, k_idx = (topk_indices == i).nonzero(as_tuple=True)

            if len(token_indices) > 0:
                # Run tokens belonging to expert i
                expert_input = x_flat[token_indices]
                expert_output = expert(expert_input)

                # Retrieve routing weights for this expert per token
                weights = topk_weights[token_indices, k_idx].unsqueeze(-1)

                # Accumulate weighted expert output into global output
                out_flat[token_indices] += weights * expert_output

        return out_flat.view(orig_shape)

In [52]:
# Test Mixture of Experts Layer
moe_layer = MoELayer(embed_dim=8, hidden_dim=16, num_experts=4, top_k=2)

# Sample input representing tokens
sample_tokens = torch.randn(2, 3, 8) # [batch, seq_len, embed_dim]
moe_output = moe_layer(sample_tokens)

print("Input shape:", sample_tokens.shape)
print("Output shape (Must match input shape):", moe_output.shape)
print("\nExpert Gating Weights Sample (First Token):\n", F.softmax(moe_layer.gate(sample_tokens[0]), dim=-1).detach().numpy())

Input shape: torch.Size([2, 3, 8])
Output shape (Must match input shape): torch.Size([2, 3, 8])

Expert Gating Weights Sample (First Token):
 [[0.30120695 0.19509143 0.16539347 0.3383081 ]
 [0.34699917 0.24990597 0.23793636 0.16515851]
 [0.09522504 0.5104047  0.27365503 0.12071527]]


### 11. Grouped-Query Attention (GQA) & Cache Compression

**Grouped-Query Attention (GQA)** is a generalization of Multi-Query Attention (MQA) and Multi-Head Attention (MHA). While MHA has an equal number of query ($Q$), key ($K$), and value ($V$) heads, and MQA has only a single $K$ and $V$ head shared by all $Q$ heads, GQA partitions query heads into groups. Each group of query heads shares a single key head and value head. This reduces memory bandwidth demands during autoregressive generation by compressing the size of the active KV cache.

In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryCausalAttention(nn.Module):
    def __init__(self, embed_dim, num_q_heads, num_kv_heads):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0, "Query heads must be divisible by KV heads"

        self.embed_dim = embed_dim
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = embed_dim // num_q_heads
        self.num_queries_per_kv = num_q_heads // num_kv_heads

        # Separate linear projections
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, kv_cache=None):
        batch_size, seq_len, _ = x.shape

        # Project inputs to Q, K, V
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # Concat KV cache if active
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            K = torch.cat([k_prev, K], dim=-2)
            V = torch.cat([v_prev, V], dim=-2)

        new_kv_cache = (K, V)
        total_seq_len = K.shape[-2]

        # Expand K and V heads to match Q heads by repeating groups
        # We repeat the KV heads to align with query heads belonging to the same group
        K_expanded = K.repeat_interleave(self.num_queries_per_kv, dim=1)
        V_expanded = V.repeat_interleave(self.num_queries_per_kv, dim=1)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K_expanded.transpose(-2, -1)) / (self.head_dim ** 0.5)

        mask = torch.ones((seq_len, total_seq_len), device=x.device)
        mask = torch.tril(mask, diagonal=total_seq_len - seq_len).view(1, 1, seq_len, total_seq_len)
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attention_weights, V_expanded)

        # Reshape output to restore back to original dimension
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        return self.out_proj(out), new_kv_cache

In [54]:
# 1. Initialize Grouped Query Attention
# Example setup: 8 query heads, 2 KV heads (compression ratio of 4x on KV size)
gqa_layer = GroupedQueryCausalAttention(embed_dim=16, num_q_heads=8, num_kv_heads=2)

# Standard input representing a prompt with 4 tokens
sample_input = torch.randn(1, 4, 16)

# Forward pass with KV cache initialization
output, cache = gqa_layer(sample_input)
print("Output shape of GQA layer:", output.shape)
print("--- Cache Size Analysis (Compression Demonstration) ---")
print("Cache Key tensor shape:", cache[0].shape)
print(f"Number of KV Heads stored in Cache: {cache[0].shape[1]} (Compared to 8 in conventional MHA)")

# Estimate compression
mha_kv_elements = 1 * 8 * 4 * (16 // 8) * 2  # [B, H_q, S, D] for K and V
gqa_kv_elements = cache[0].nelement() + cache[1].nelement()
compression_pct = (1 - (gqa_kv_elements / mha_kv_elements)) * 100
print(f"KV Cache Element count: MHA = {mha_kv_elements} vs. GQA = {gqa_kv_elements}")
print(f"KV Cache memory footprint reduction: {compression_pct:.1f}%")

Output shape of GQA layer: torch.Size([1, 4, 16])
--- Cache Size Analysis (Compression Demonstration) ---
Cache Key tensor shape: torch.Size([1, 2, 4, 2])
Number of KV Heads stored in Cache: 2 (Compared to 8 in conventional MHA)
KV Cache Element count: MHA = 128 vs. GQA = 32
KV Cache memory footprint reduction: 75.0%


### 12. Recurrent Language Model (RNN/GRU) vs. GPT

Unlike Transformers which process all tokens in parallel during training (using causal masks), recurrent models process tokens sequentially step-by-step ($h_t = f(h_{t-1}, x_t)$). Here we implement a custom **Gated Recurrent Unit (GRU) Language Model** to compare its architecture, computational properties, and scaling characteristics with our `MiniGPT`.

In [55]:
class RecurrentLM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)

        # GRU acts as our recurrent processor sequence bottleneck
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, idx, hidden=None):
        # Embedded shape: [batch_size, seq_len, embed_dim]
        x = self.token_embeddings(idx)

        # Recurrent sequential update
        out, hidden = self.rnn(x, hidden)

        # Map hidden state outputs back to logits
        logits = self.lm_head(out)
        return logits, hidden

In [56]:
# 1. Initialize and Pretrain our Recurrent Language Model on the same data
rnn_model = RecurrentLM(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    hidden_dim=32,
    num_layers=2
)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(rnn_model.parameters(), lr=0.01)

rnn_model.train()
print("--- Training Recurrent Language Model ---")
for epoch in range(150):
    optimizer.zero_grad()
    # We pass inputs sequentially
    logits, _ = rnn_model(X_train)
    loss = criterion(logits.view(-1, len(tokenizer.vocab)), Y_train.view(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:03d} / 150 | Recurrent LM Loss: {loss.item():.4f}")

--- Training Recurrent Language Model ---
Epoch 050 / 150 | Recurrent LM Loss: 0.0205
Epoch 100 / 150 | Recurrent LM Loss: 0.0047
Epoch 150 / 150 | Recurrent LM Loss: 0.0027


#### Architectural Comparison Table

| Feature | GPT (Transformer Decoder) | Recurrent Language Model (RNN/GRU) |
| :--- | :--- | :--- |
| **Training Computation** | Parallel over sequence dimension ($O(1)$ sequential operations) | Sequential step-by-step ($O(N)$ sequential operations) |
| **Inference State Size** | Dynamic KV Cache grows with sequence length ($O(N)$ memory) | Constant Hidden State size ($O(1)$ memory) |
| **Long-range Dependencies** | Direct $O(1)$ routing via self-attention layers | Lossy bottleneck through recurrent states (vanishing/exploding gradients) |

### 13. Investigating Expert Specialization in MoE

To see if the Gating Network learns to specialize or route different concepts/features to specific experts, we can analyze the routing decisions (top-$k$ indices) for several distinct inputs representing different token types.

In [57]:
import torch
import torch.nn.functional as F

# Let's inspect routing distributions for 3 distinct test categories of tokens
# Each category is represented by a cluster of embeddings in the latent space
torch.manual_seed(42)

# Simulated token categories:
# Category A (e.g., verbs/actions), Category B (e.g., nouns/objects), Category C (e.g., punctuation)
c_A = torch.randn(5, 8) + torch.tensor([2.0, -2.0, 1.0, 0.0, -1.0, 2.0, 0.5, -0.5])
c_B = torch.randn(5, 8) + torch.tensor([-2.0, 2.0, -1.0, 1.0, 2.0, -2.0, -0.5, 0.5])
c_C = torch.randn(5, 8) * 0.1  # near zero

test_tokens = torch.cat([c_A, c_B, c_C], dim=0) # Shape: [15, 8]

# Query routing gates
with torch.no_grad():
    logits = moe_layer.gate(test_tokens)
    probs = F.softmax(logits, dim=-1)
    topk_vals, topk_indices = torch.topk(logits, k=2, dim=-1)

# Print expert routing decisions
print("--- Routing Decisions for Token Clusters ---")
for idx, (token, indices, p) in enumerate(zip(test_tokens, topk_indices, probs)):
    category = "Verbs (A)" if idx < 5 else "Nouns (B)" if idx < 10 else "Punctuation (C)"
    print(f"Token {idx:02d} | Category: {category:<15} | Selected Experts: {indices.tolist()} | Softmax: {p.numpy().round(3)}")

--- Routing Decisions for Token Clusters ---
Token 00 | Category: Verbs (A)       | Selected Experts: [2, 3] | Softmax: [0.118 0.168 0.4   0.314]
Token 01 | Category: Verbs (A)       | Selected Experts: [2, 0] | Softmax: [0.18  0.162 0.499 0.159]
Token 02 | Category: Verbs (A)       | Selected Experts: [2, 1] | Softmax: [0.092 0.14  0.685 0.084]
Token 03 | Category: Verbs (A)       | Selected Experts: [2, 3] | Softmax: [0.083 0.207 0.479 0.232]
Token 04 | Category: Verbs (A)       | Selected Experts: [1, 3] | Softmax: [0.18  0.408 0.085 0.328]
Token 05 | Category: Nouns (B)       | Selected Experts: [1, 3] | Softmax: [0.242 0.357 0.075 0.327]
Token 06 | Category: Nouns (B)       | Selected Experts: [1, 0] | Softmax: [0.3   0.428 0.115 0.157]
Token 07 | Category: Nouns (B)       | Selected Experts: [0, 3] | Softmax: [0.529 0.131 0.161 0.178]
Token 08 | Category: Nouns (B)       | Selected Experts: [1, 0] | Softmax: [0.22  0.512 0.083 0.185]
Token 09 | Category: Nouns (B)       | Selecte

### 14. Shared Latent Workspace and Dynamic Routing

To break away from the quadratic complexity of standard self-attention, we can implement a **Shared Latent Workspace** (Global Memory Bottleneck). Here, we instantiate a small set of learned latent vectors ("workspace").
1. **Write (Cross-Attention)**: Input tokens dynamically route/write their information into the shared latent workspace.
2. **Read (Cross-Attention)**: Input tokens query the updated workspace to retrieve global context. This creates an $O(N \times L)$ bottleneck where $L \ll N$ (number of latents).

In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossAttentionBottleneck(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, query, key_val):
        # query: [B, Target_Len, D]
        # key_val: [B, Source_Len, D]
        B, Target_Len, _ = query.shape
        _, Source_Len, _ = key_val.shape

        Q = self.q_proj(query).view(B, Target_Len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_val).view(B, Source_Len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_val).view(B, Source_Len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)

        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, Target_Len, self.embed_dim)
        return self.out_proj(out), attn_weights

class SharedWorkspaceTransformer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_latents):
        super().__init__()
        self.num_latents = num_latents
        # Learnable latent workspace vectors
        self.latent_workspace = nn.Parameter(torch.randn(1, num_latents, embed_dim))

        # Cross-attention handlers to write and read
        self.write_attn = CrossAttentionBottleneck(embed_dim, num_heads)
        self.read_attn = CrossAttentionBottleneck(embed_dim, num_heads)

        self.norm_tokens = nn.LayerNorm(embed_dim)
        self.norm_latents = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, N, D = x.shape
        # Expand latent workspace for current batch
        latents = self.latent_workspace.expand(B, -1, -1)

        # Step 1: Write (Tokens -> Latent Workspace)
        # Latents act as Query, Tokens as Key/Value
        updated_latents, write_weights = self.write_attn(self.norm_latents(latents), self.norm_tokens(x))
        latents = latents + updated_latents

        # Step 2: Read (Latent Workspace -> Tokens)
        # Tokens act as Query, Latents as Key/Value
        updated_tokens, read_weights = self.read_attn(self.norm_tokens(x), self.norm_latents(latents))
        x = x + updated_tokens

        return x, write_weights, read_weights

In [59]:
# Test the Shared Latent Workspace
workspace_module = SharedWorkspaceTransformer(embed_dim=16, num_heads=2, num_latents=3)

tokens_in = torch.randn(2, 10, 16) # Batch size of 2, 10 tokens, 16 embedding size
tokens_out, write_w, read_w = workspace_module(tokens_in)

print("Input Tokens Shape:", tokens_in.shape)
print("Output Tokens Shape:", tokens_out.shape)
print("Write (Tokens to Workspace) Attention Shape (Batch, Heads, Latents, Tokens):", write_w.shape)
print("Read (Workspace to Tokens) Attention Shape (Batch, Heads, Tokens, Latents):", read_w.shape)

Input Tokens Shape: torch.Size([2, 10, 16])
Output Tokens Shape: torch.Size([2, 10, 16])
Write (Tokens to Workspace) Attention Shape (Batch, Heads, Latents, Tokens): torch.Size([2, 2, 3, 10])
Read (Workspace to Tokens) Attention Shape (Batch, Heads, Tokens, Latents): torch.Size([2, 2, 10, 3])


### 15. Multimodality: Image Patchifier from Scratch

To bridge vision and language models, Vision Transformers (ViTs) slice raw images into small sequential patches. Here we implement three equivalent methods to flatten and project an image of shape `[B, C, H, W]` into sequence patches of shape `[B, Num_Patches, Patch_Dim]`:
1. **Method 1**: Tensor Reshaping (`reshape` + `permute`)
2. **Method 2**: PyTorch's `unfold` operator
3. **Method 3**: Projection via a 2D Convolution (`Conv2d` with kernel size equal to patch size)

In [60]:
import torch
import torch.nn as nn

class ImagePatchifier(nn.Module):
    def __init__(self, patch_size=4, in_channels=3, embed_dim=16):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        self.patch_dim = in_channels * patch_size * patch_size

        # Projection layer to map raw flat patches to embed_dim (for Methods 1 & 2)
        self.proj_linear = nn.Linear(self.patch_dim, embed_dim)

        # Conv2d projection (Method 3) directly does slicing and projection simultaneously
        self.proj_conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def patchify_reshape(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        P = self.patch_size
        num_patches_h = H // P
        num_patches_w = W // P

        # Reshape to chunk dimensions
        # [B, C, num_patches_h, P, num_patches_w, P]
        patches = x.view(B, C, num_patches_h, P, num_patches_w, P)
        # Permute to move patch spatial dims together: [B, num_patches_h, num_patches_w, P, P, C]
        patches = patches.permute(0, 2, 4, 3, 5, 1).contiguous()
        # Flatten spatial patches: [B, Num_Patches, Patch_Dim]
        flat_patches = patches.view(B, num_patches_h * num_patches_w, self.patch_dim)

        return self.proj_linear(flat_patches)

    def patchify_unfold(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        P = self.patch_size

        # Unfold along height and width
        patches = x.unfold(2, P, P).unfold(3, P, P)
        # Shape is now [B, C, H_patches, W_patches, P, P]
        patches = patches.permute(0, 2, 3, 4, 5, 1).contiguous()
        flat_patches = patches.view(B, -1, self.patch_dim)

        return self.proj_linear(flat_patches)

    def patchify_conv(self, x):
        # x: [B, C, H, W]
        # Conv projection computes [B, embed_dim, H_patches, W_patches]
        conv_out = self.proj_conv(x)
        # Flatten spatial dimensions and transpose to [B, Num_Patches, embed_dim]
        flat_patches = conv_out.flatten(2).transpose(1, 2)
        return flat_patches

In [61]:
# Test Patchifiers
patch_size = 4
patchifier = ImagePatchifier(patch_size=patch_size, in_channels=3, embed_dim=16)

# Match weights of Conv projection with linear projection for equivalence check
with torch.no_grad():
    # Flatten Conv weight tensor to map perfectly to linear projection weights
    patchifier.proj_linear.weight.copy_(patchifier.proj_conv.weight.view(16, -1))
    patchifier.proj_linear.bias.copy_(patchifier.proj_conv.bias)

# Batch of 1 image, 3 channels, size 12x12
sample_image = torch.randn(1, 3, 12, 12)

out_reshape = patchifier.patchify_reshape(sample_image)
out_unfold = patchifier.patchify_unfold(sample_image)
out_conv = patchifier.patchify_conv(sample_image)

print("Input Image Shape:", sample_image.shape)
print("Method 1 (Reshape) Shape:", out_reshape.shape)
print("Method 2 (Unfold) Shape: ", out_unfold.shape)
print("Method 3 (Conv2D) Shape: ", out_conv.shape)

# Verification check
print("\nIs Reshape equivalent to Unfold?:", torch.allclose(out_reshape, out_unfold, atol=1e-5))
print("Is Reshape equivalent to Conv2D?:", torch.allclose(out_reshape, out_conv, atol=1e-5))

Input Image Shape: torch.Size([1, 3, 12, 12])
Method 1 (Reshape) Shape: torch.Size([1, 9, 16])
Method 2 (Unfold) Shape:  torch.Size([1, 9, 16])
Method 3 (Conv2D) Shape:  torch.Size([1, 9, 16])

Is Reshape equivalent to Unfold?: True
Is Reshape equivalent to Conv2D?: False


### 16. Vision Transformer (ViT) Encoder Block

To build a complete **Vision Transformer (ViT)**, we add a few extra components to our patchifier:
1. **CLS Token**: A learnable token prepended to the sequence of patch embeddings to capture global image representation.
2. **Position Embeddings**: Learnable 1D parameters added to each patch representation to retain spatial position context.
3. **Bidirectional Self-Attention**: Unlike decoders which require causal masking, vision encoders use bidirectional attention (tokens can attend to both past and future positions).

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ViTAttention(nn.Module):
    """Standard Bidirectional Self-Attention (no causal mask)"""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # Softmax applies globally over all positions (bidirectional)
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, N, D)
        return self.out_proj(out)

class VisionTransformerEncoder(nn.Module):
    def __init__(self, num_patches, embed_dim, num_heads, hidden_dim):
        super().__init__()
        # Learnable classification token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable position embeddings (num_patches + 1 for CLS token)
        self.pos_embeddings = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = ViTAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, patch_embeddings):
        B = patch_embeddings.shape[0]

        # Prepend CLS Token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, patch_embeddings], dim=1)

        # Add Position Embeddings
        x = x + self.pos_embeddings

        # Encoder Layer Block
        attn_out = x + self.attn(self.ln1(x))
        x = attn_out + self.ffn(self.ln2(attn_out))
        return x

In [63]:
# Test Vision Transformer Pipeline
# Slicing a 12x12 image into 3x3 patches (patch_size=4) -> 9 patches
patch_size = 4
img = torch.randn(2, 3, 12, 12)

# Step 1: Patchify & Project Image
patch_processor = ImagePatchifier(patch_size=patch_size, in_channels=3, embed_dim=16)
patches = patch_processor.patchify_conv(img)  # Shape: [2, 9, 16]

# Step 2: Pass through Vision Transformer
vit_block = VisionTransformerEncoder(num_patches=9, embed_dim=16, num_heads=2, hidden_dim=32)
encoded_representation = vit_block(patches)

print("Input Patches Shape:", patches.shape)
print("Encoded Sequence Output Shape (Includes prepended CLS Token):", encoded_representation.shape)
print("CLS Token vector representation:", encoded_representation[:, 0].shape)

Input Patches Shape: torch.Size([2, 9, 16])
Encoded Sequence Output Shape (Includes prepended CLS Token): torch.Size([2, 10, 16])
CLS Token vector representation: torch.Size([2, 16])


### 17. Mini-CLIP: Contrastive Language-Image Pretraining

**CLIP** (Contrastive Language-Image Pretraining) maps images and text captions into a joint, multi-modal embedding space. We train both encoders simultaneously using a symmetric **contrastive loss**:
1. For a batch of $B$ matching image-text pairs, we construct a similarity matrix of size $B \times B$.
2. We maximize the similarity along the diagonal (matching pairs) while minimizing similarity off the diagonal (negative pairings) using cross-entropy loss applied row-wise and column-wise.

In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MiniCLIP(nn.Module):
    def __init__(self, text_model, image_encoder, embed_dim, projection_dim=16):
        super().__init__()
        self.text_encoder = text_model
        self.image_encoder = image_encoder

        # Projection layers to map different embedding dimensions to a shared latent space
        self.text_projection = nn.Linear(text_model.token_embeddings.embedding_dim, projection_dim, bias=False)
        self.image_projection = nn.Linear(image_encoder.ffn[-1].out_features, projection_dim, bias=False)

        # Learnable temperature parameter scaling similarity scores
        self.logit_scale = nn.Parameter(torch.ones([]) * log(1 / 0.07) if 'log' in globals() else torch.tensor(2.659))

    def get_image_features(self, pixel_values, patch_processor):
        # 1. Slice image into patches
        patches = patch_processor.patchify_conv(pixel_values)
        # 2. Extract global image features using the CLS token output (index 0)
        image_outputs = self.image_encoder(patches)
        cls_embeddings = image_outputs[:, 0]
        # 3. Project to shared space and normalize
        image_latents = self.image_projection(cls_embeddings)
        return F.normalize(image_latents, p=2, dim=-1)

    def get_text_features(self, text_ids):
        # 1. Forward pass through our text model (using last sequence output representation)
        text_outputs, _ = self.text_encoder(text_ids)
        sentence_embeddings = text_outputs[:, -1]
        # 2. Project to shared space and normalize
        text_latents = self.text_projection(sentence_embeddings)
        return F.normalize(text_latents, p=2, dim=-1)

    def forward(self, text_ids, pixel_values, patch_processor):
        # Extract normalized representations
        image_embeds = self.get_image_features(pixel_values, patch_processor)
        text_embeds = self.get_text_features(text_ids)

        # Compute cosine similarity matrix
        t_scale = self.logit_scale.exp()
        logits_per_image = torch.matmul(image_embeds, text_embeds.t()) * t_scale
        logits_per_text = logits_per_image.t()

        # Symmetric loss labels (each matching index is its target index along diagonal)
        labels = torch.arange(image_embeds.size(0), device=text_ids.device)

        loss_i = F.cross_entropy(logits_per_image, labels)
        loss_t = F.cross_entropy(logits_per_text, labels)

        return (loss_i + loss_t) / 2

In [65]:
# Fix MiniCLIP to extract hidden representation instead of vocabulary logits
import torch
import torch.nn as nn
import torch.nn.functional as F

# Override MiniCLIP's text features retrieval to obtain embeddings of dimension 16
class FixedMiniCLIP(MiniCLIP):
    def get_text_features(self, text_ids):
        # 1. Forward pass through token embeddings and block layers (bypassing the final lm_head)
        positions = torch.arange(0, text_ids.shape[1], device=text_ids.device).unsqueeze(0)
        x = self.text_encoder.token_embeddings(text_ids) + self.text_encoder.position_embeddings(positions)
        for block in self.text_encoder.blocks:
            x, _ = block(x)
        x = self.text_encoder.ln_f(x)

        # Use representation from the last sequence token (shape: [Batch, Embed_Dim])
        sentence_embeddings = x[:, -1]

        # 2. Project to shared space and normalize
        text_latents = self.text_projection(sentence_embeddings)
        return F.normalize(text_latents, p=2, dim=-1)

# 1. Initialize our subnetworks
vit_encoder = VisionTransformerEncoder(num_patches=9, embed_dim=16, num_heads=2, hidden_dim=32)
patch_proc = ImagePatchifier(patch_size=4, in_channels=3, embed_dim=16)

# 2. Assemble the fixed CLIP model
clip_model = FixedMiniCLIP(gpt_model, vit_encoder, embed_dim=16, projection_dim=8)
optimizer = torch.optim.Adam(clip_model.parameters(), lr=0.01)

# 3. Dummy batch of matching image-text samples
sample_texts = torch.randint(1, len(tokenizer.vocab), (3, 8)) # 3 text tokens
sample_imgs = torch.randn(3, 3, 12, 12) # 3 matching images

# Single contrastive training step
clip_model.train()
optimizer.zero_grad()
loss = clip_model(sample_texts, sample_imgs, patch_proc)
loss.backward()
optimizer.step()

print("Contrastive Loss output:", loss.item())
print("Shared Projection Embeddings shape (Image):", clip_model.get_image_features(sample_imgs, patch_proc).shape)

Contrastive Loss output: 2.39013409614563
Shared Projection Embeddings shape (Image): torch.Size([3, 8])


### 18. Linear Vision Adapter: Bridging Frozen Encoders

To construct a multimodal model (like LLaVA), we connect a pre-trained, frozen vision encoder to a pre-trained, frozen language model.

Since the raw output dimension of the vision encoder (ViT) does not necessarily match the input token embedding dimension of the Language Model (GPT), we train a **Linear Vision Adapter** (a simple projection layer) to map the visual tokens into the language space.

In [66]:
import torch
import torch.nn as nn

class LinearVisionAdapter(nn.Module):
    def __init__(self, vision_dim, text_dim):
        super().__init__()
        # A single linear layer mapping visual token vectors to language embedding dimensions
        self.adapter = nn.Linear(vision_dim, text_dim)

    def forward(self, vision_outputs):
        # vision_outputs shape: [Batch, Num_Patches, Vision_Dim]
        # returns projected embeddings: [Batch, Num_Patches, Text_Dim]
        return self.adapter(vision_outputs)

In [67]:
# 1. Freeze our base models (Vision Encoder & GPT Language Model)
for param in vit_encoder.parameters():
    param.requires_grad = False

for param in gpt_model.parameters():
    param.requires_grad = False

# 2. Instantiate our trainable projection adapter
# Connecting vit_encoder output dimension (16) to gpt_model embedding dimension (16)
vision_adapter = LinearVisionAdapter(vision_dim=16, text_dim=16)

# 3. Simulate multimodal forward pass
sample_img = torch.randn(1, 3, 12, 12)

# Step A: Get visual patches, run through frozen ViT
img_patches = patch_proc.patchify_conv(sample_img) # [1, 9, 16]
visual_features = vit_encoder(img_patches)          # [1, 10, 16] (includes CLS token)

# Step B: Project visual features to language space using the adapter
projected_visual_tokens = vision_adapter(visual_features) # [1, 10, 16]

# Step C: Get text embeddings from frozen language model
sample_text_ids = torch.tensor([[tokenizer.vocab['learning'], tokenizer.vocab['embeddings']]]) # [1, 2]
word_embeddings = gpt_model.token_embeddings(sample_text_ids) # [1, 2, 16]

# Step D: Concatenate visual tokens as a prefix to the text tokens
multimodal_prompt = torch.cat([projected_visual_tokens, word_embeddings], dim=1) # [1, 12, 16]

print("Projected Visual Tokens shape:", projected_visual_tokens.shape)
print("Text Tokens embedding shape:", word_embeddings.shape)
print("Combined Multimodal input sequence:", multimodal_prompt.shape)
print("\n--- Verification ---")
print("Is Vision Encoder frozen?:", all(not p.requires_grad for p in vit_encoder.parameters()))
print("Is GPT Model frozen?:", all(not p.requires_grad for p in gpt_model.parameters()))
print("Is Vision Adapter trainable?:", all(p.requires_grad for p in vision_adapter.parameters()))

Projected Visual Tokens shape: torch.Size([1, 10, 16])
Text Tokens embedding shape: torch.Size([1, 2, 16])
Combined Multimodal input sequence: torch.Size([1, 12, 16])

--- Verification ---
Is Vision Encoder frozen?: True
Is GPT Model frozen?: True
Is Vision Adapter trainable?: True


### 19. MLP Vision Adapter: Non-linear Projection

While a linear adapter is computationally simple, a **Multi-Layer Perceptron (MLP) Vision Adapter** adds non-linear capability (e.g., LayerNorm -> Linear -> GELU -> Linear). This extra expressiveness allows the model to align complex visual feature structures more smoothly with the text embedding space.

In [68]:
import torch
import torch.nn as nn

class MLPVisionAdapter(nn.Module):
    def __init__(self, vision_dim, text_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(vision_dim),
            nn.Linear(vision_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, text_dim)
        )

    def forward(self, vision_outputs):
        # vision_outputs: [Batch, Num_Patches, Vision_Dim]
        # returns projected embeddings: [Batch, Num_Patches, Text_Dim]
        return self.net(vision_outputs)

In [69]:
# 1. Initialize both adapters
linear_adapter = LinearVisionAdapter(vision_dim=16, text_dim=16)
mlp_adapter = MLPVisionAdapter(vision_dim=16, text_dim=16, hidden_dim=32)

# 2. Extract visual features for comparison
with torch.no_grad():
    sample_img_patches = patch_proc.patchify_conv(sample_img)
    vis_features = vit_encoder(sample_img_patches)

    # Project using both methods
    linear_projected = linear_adapter(vis_features)
    mlp_projected = mlp_adapter(vis_features)

# 3. Analyze differences
print("Input Visual Features Shape:", vis_features.shape)
print("Linear Adapter Projected Shape:", linear_projected.shape)
print("MLP Adapter Projected Shape:", mlp_projected.shape)
print("\n--- Parameter Complexity Comparison ---")
linear_params = sum(p.numel() for p in linear_adapter.parameters())
mlp_params = sum(p.numel() for p in mlp_adapter.parameters())
print(f"Linear Adapter: {linear_params} trainable parameters (1 Matrix mapping)")
print(f"MLP Adapter:    {mlp_params} trainable parameters (Multi-layer non-linear pipeline)")

Input Visual Features Shape: torch.Size([1, 10, 16])
Linear Adapter Projected Shape: torch.Size([1, 10, 16])
MLP Adapter Projected Shape: torch.Size([1, 10, 16])

--- Parameter Complexity Comparison ---
Linear Adapter: 272 trainable parameters (1 Matrix mapping)
MLP Adapter:    1104 trainable parameters (Multi-layer non-linear pipeline)


### 20. Multimodal Visual Instruction Tuning with LoRA

To perform **Visual Instruction Tuning**, we will assemble an end-to-end Multimodal LLM pipeline:
1. **Frozen ViT Encoder + Trainable MLP Adapter**: Slices an input image into patches, encodes them, and projects them into the word embedding space.
2. **LoRA-wrapped Language Model**: Wraps the query/key projections in our mini-decoder block with low-rank adapters (`LoRALinear`) to keep base parameters frozen while adapting efficiently.
3. **Instruction Dataset**: Sets up simulated multi-modal instructions for captioning, counting, and question-answering tasks.
4. **Optimization**: Optimizes both the MLP Vision Adapter and the LoRA parameters using a next-token cross-entropy objective.

In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Prepare instruction-tuning corpus
instruction_data = [
    # (Image representation key, Prompt, Target Answer)
    ("cat_img", "describe image", "cat sat on"),
    ("neural_img", "count objects", "two objects"),
    ("deep_img", "what is this", "deep learning")
]

# Let's fit our tokenizer with these new multimodal target answers
all_texts = [item[1] for item in instruction_data] + [item[2] for item in instruction_data]
tokenizer.fit(all_texts)

vocab_size = len(tokenizer.vocab)
print("Instruction Tokenizer Vocab Size:", vocab_size)
print("Vocab Map:", tokenizer.vocab)

Instruction Tokenizer Vocab Size: 29
Vocab Map: {'<PAD>': 0, '<UNK>': 1, 'and': 2, 'building': 3, 'deep': 4, 'embeddings': 5, 'fascinating': 6, 'is': 7, 'it': 8, 'language': 9, 'learning': 10, 'love': 11, 'make': 12, 'natural': 13, 'networks': 14, 'neural': 15, 'powerful': 16, 'processing': 17, 'we': 18, 'cat': 19, 'count': 20, 'describe': 21, 'image': 22, 'objects': 23, 'on': 24, 'sat': 25, 'this': 26, 'two': 27, 'what': 28}


In [71]:
class LoRAMultimodalLLM(nn.Module):
    def __init__(self, base_gpt, vision_encoder, vision_dim, text_dim, r=4):
        super().__init__()
        self.vision_encoder = vision_encoder
        # Keep vision encoder strictly frozen
        for p in self.vision_encoder.parameters():
            p.requires_grad = False

        # Trainable MLP Adapter
        self.mlp_adapter = MLPVisionAdapter(vision_dim=vision_dim, text_dim=text_dim, hidden_dim=32)

        # Clone and wrap the base GPT blocks with LoRA
        self.gpt = base_gpt
        for block in self.gpt.blocks:
            # Wrap Q & K projection in attention with LoRA
            block.attn.q_proj = LoRALinear(block.attn.q_proj, r=r)
            block.attn.k_proj = LoRALinear(block.attn.k_proj, r=r)

    def forward(self, images, instruction_ids):
        # 1. Project image patches: [Batch, Num_Patches, Text_Dim]
        # We simulate visual feature extraction
        vis_feats = self.vision_encoder(images)
        projected_vis = self.mlp_adapter(vis_feats)

        # 2. Get text embeddings: [Batch, Prompt_Len, Text_Dim]
        text_embeds = self.gpt.token_embeddings(instruction_ids)

        # 3. Concatenate visual prefixes with instruction tokens
        multimodal_inputs = torch.cat([projected_vis, text_embeds], dim=1)

        # 4. Generate positional coordinates over the unified length
        batch_size, total_seq_len, embed_dim = multimodal_inputs.shape
        positions = torch.arange(0, total_seq_len, device=instruction_ids.device).unsqueeze(0)

        x = multimodal_inputs + self.gpt.position_embeddings(positions)

        # Forward pass through decoder layers
        for block in self.gpt.blocks:
            x, _ = block(x)

        x = self.gpt.ln_f(x)
        logits = self.gpt.lm_head(x)
        return logits

In [72]:
# Initialize components
base_gpt = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=24  # Expand sequence context for concatenated vision + text
)

# Vision components
vit_encoder = VisionTransformerEncoder(num_patches=9, embed_dim=16, num_heads=2, hidden_dim=32)
patch_proc = ImagePatchifier(patch_size=4, in_channels=3, embed_dim=16)

# Construct unified Multi-Modal Model
multimodal_model = LoRAMultimodalLLM(base_gpt, vit_encoder, vision_dim=16, text_dim=16, r=4)

# Define Optimizer targeting ONLY trainable parameters (MLP Adapter + LoRA variables)
optim_params = [p for p in multimodal_model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(optim_params, lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("--- Model Parameter Audit ---")
all_params = sum(p.numel() for p in multimodal_model.parameters())
train_params = sum(p.numel() for p in optim_params)
print(f"Total parameters in system: {all_params}")
print(f"Active trainable parameters (MLP Adapter + LoRA layers): {train_params} ({100 * train_params / all_params:.2f}%)")

--- Model Parameter Audit ---
Total parameters in system: 10464
Active trainable parameters (MLP Adapter + LoRA layers): 7104 (67.89%)


In [73]:
# Simulated visual dataset generation
img_dict = {
    "cat_img": torch.randn(1, 3, 12, 12),
    "neural_img": torch.randn(1, 3, 12, 12),
    "deep_img": torch.randn(1, 3, 12, 12)
}

# Training Loop over tasks
multimodal_model.train()
print("--- Visual Instruction Tuning Progress ---")
for epoch in range(120):
    epoch_loss = 0.0
    for img_key, prompt_text, target_text in instruction_data:
        optimizer.zero_grad()

        # Load image & tokenize prompt & target sequence
        img_tensor = patch_proc.patchify_conv(img_dict[img_key])

        # Create prompt + target sequence: e.g. "describe image <answer>"
        full_text = f"{prompt_text} {target_text}"
        encoded = tokenizer.encode(full_text, max_length=12)

        # Shifting targets: we calculate next-token loss on the text part
        input_ids = torch.tensor([encoded[:-1]], dtype=torch.long)
        target_ids = torch.tensor([encoded[1:]], dtype=torch.long)

        # Forward pass through Multimodal model
        logits = multimodal_model(img_tensor, input_ids)

        # Evaluate only the target response tokens (ignoring visual token outputs in sequence prefix)
        # Shape of logits is [1, Num_Patches + Prompt_Seq_Len, Vocab_Size]
        # We exclude all 10 visual patch projection steps (9 spatial patches + 1 CLS token)
        text_logits = logits[:, 10:]

        loss = criterion(text_logits.view(-1, len(tokenizer.vocab)), target_ids.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:03d} / 120 | Multimodal CrossEntropy Loss: {epoch_loss/3:.4f}")

--- Visual Instruction Tuning Progress ---
Epoch 030 / 120 | Multimodal CrossEntropy Loss: 0.0186
Epoch 060 / 120 | Multimodal CrossEntropy Loss: 0.0058
Epoch 090 / 120 | Multimodal CrossEntropy Loss: 0.0030
Epoch 120 / 120 | Multimodal CrossEntropy Loss: 0.0018


In [74]:
# Evaluate instruction tuning outputs
multimodal_model.eval()
with torch.no_grad():
    print("\n--- Evaluating Multimodal Instructions ---")
    for img_key, prompt_text, target_text in instruction_data:
        img_tensor = patch_proc.patchify_conv(img_dict[img_key])
        prompt_ids = torch.tensor([tokenizer.encode(prompt_text, max_length=4)], dtype=torch.long)

        logits = multimodal_model(img_tensor, prompt_ids)
        # Strip visual token offsets to isolate text logits
        text_logits = logits[:, 9:]
        predicted_ids = torch.argmax(text_logits, dim=-1)[0].tolist()

        print(f"\n[Task: {prompt_text}]")
        print("Target:", target_text)
        print("Model Prediction:", tokenizer.decode(predicted_ids))


--- Evaluating Multimodal Instructions ---

[Task: describe image]
Target: cat sat on
Model Prediction: deep image cat learning learning

[Task: count objects]
Target: two objects
Model Prediction: deep objects two learning learning

[Task: what is this]
Target: deep learning
Model Prediction: deep is this deep learning


### 21. Q-Former-Inspired Resampler

To scale vision-language models efficiently (especially with high-resolution images yielding many patches), we can implement a **Q-Former-inspired Resampler** (similar to BLIP-2).

Instead of passing all raw image patch tokens directly into the LLM, the Resampler uses a fixed number of **learnable query embeddings** that interact with the visual tokens via cross-attention. This compresses any variable sequence length of raw visual features into a fixed-length set of highly informative visual bottleneck representations.

In [75]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class QFormerResampler(nn.Module):
    def __init__(self, num_queries=4, embed_dim=16, num_heads=2, hidden_dim=32):
        super().__init__()
        # Fixed set of learnable latent query tokens
        self.queries = nn.Parameter(torch.randn(1, num_queries, embed_dim))

        # Cross-Attention layer: Queries attend to Image Features (Keys/Values)
        self.cross_attn = CrossAttentionBottleneck(embed_dim=embed_dim, num_heads=num_heads)
        self.norm_queries = nn.LayerNorm(embed_dim)
        self.norm_visuals = nn.LayerNorm(embed_dim)

        # Feed Forward block for the queries
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm_ffn = nn.LayerNorm(embed_dim)

    def forward(self, visual_features):
        # visual_features: [Batch, Num_Patches, Embed_Dim]
        batch_size = visual_features.shape[0]

        # Expand query tokens to batch size
        q = self.queries.expand(batch_size, -1, -1) # [Batch, Num_Queries, Embed_Dim]

        # 1. Cross-Attention: Queries (Q) attend to Visual Features (K, V)
        attn_out, weights = self.cross_attn(self.norm_queries(q), self.norm_visuals(visual_features))
        q = q + attn_out

        # 2. Feed-forward update
        q = q + self.ffn(self.norm_ffn(q))

        return q, weights

In [76]:
# Verification and Forward Pass through our Q-Former Resampler
resampler = QFormerResampler(num_queries=4, embed_dim=16, num_heads=2, hidden_dim=32)

# Simulating high-resolution image generating 16 patches (instead of 9)
simulated_large_visuals = torch.randn(2, 16, 16) # Batch=2, Patches=16, Dim=16

# Downsample using Q-Former Resampler
bottleneck_features, attention_weights = resampler(simulated_large_visuals)

print("Input Visual Features Shape (representing 16 patches):", simulated_large_visuals.shape)
print("Compressed Bottleneck Output Shape (fixed query size):", bottleneck_features.shape)
print("Cross-Attention Weights Shape (Batch, Heads, Queries, Patches):", attention_weights.shape)
print(f"Sequence length reduction ratio: {simulated_large_visuals.shape[1]} tokens -> {bottleneck_features.shape[1]} tokens")

Input Visual Features Shape (representing 16 patches): torch.Size([2, 16, 16])
Compressed Bottleneck Output Shape (fixed query size): torch.Size([2, 4, 16])
Cross-Attention Weights Shape (Batch, Heads, Queries, Patches): torch.Size([2, 2, 4, 16])
Sequence length reduction ratio: 16 tokens -> 4 tokens


### 22. Gated Cross-Attention (Flamingo-style Visual Injection)

Instead of prepending visual tokens as a prefix to the text input, architectures like **Flamingo** inject visual memory directly into the intermediate layers of the Transformer decoder using **Gated Cross-Attention**.

Key features:
1. **Cross-Attention**: The text sequence acts as the Query ($Q$), while the visual features act as the Key ($K$) and Value ($V$).
2. **Tanh Gating**: A learnable parameter $\tanh(\alpha)$ gates the output of the cross-attention block. By initializing $\alpha = 0$, the gate is completely closed at the start of training, ensuring that the pre-trained text-only capabilities are fully preserved at initialization.

In [77]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedCrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.cross_attn = CrossAttentionBottleneck(embed_dim=embed_dim, num_heads=num_heads)
        self.norm = nn.LayerNorm(embed_dim)

        # Learnable gating parameter initialized to zero
        self.tanh_gate = nn.Parameter(torch.zeros(1))

    def forward(self, x, visual_features):
        # x (text embeddings): [Batch, Text_Len, Embed_Dim]
        # visual_features: [Batch, Vis_Len, Embed_Dim]

        # Standard cross-attention where text queries the visuals
        attn_out, weights = self.cross_attn(self.norm(x), visual_features)

        # Apply tanh gating: gate output is 0 at initialization
        out = x + torch.tanh(self.tanh_gate) * attn_out
        return out, weights

In [78]:
class GatedMultimodalTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_features):
        super().__init__()
        # Base text-only operations
        self.ln1 = RMSNorm(embed_dim)
        self.self_attn = MultiHeadCausalAttention(embed_dim, num_heads)

        # Visual injection cross-attention
        self.gated_cross_attn = GatedCrossAttention(embed_dim, num_heads)

        self.ln2 = RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, hidden_features)

    def forward(self, x, visual_features):
        # 1. Standard Self-Attention over Text Tokens
        attn_out, self_weights = self.self_attn(self.ln1(x))
        x = x + attn_out

        # 2. Gated Cross-Attention attending to Visual Features
        x, cross_weights = self.gated_cross_attn(x, visual_features)

        # 3. Feed Forward network
        x = x + self.ffn(self.ln2(x))
        return x, self_weights, cross_weights

In [79]:
# Verification and Forward Pass
gated_block = GatedMultimodalTransformerBlock(embed_dim=16, num_heads=2, hidden_features=32)

sim_text = torch.randn(2, 5, 16)       # Batch=2, Sequence Length=5, Dim=16
sim_visuals = torch.randn(2, 4, 16)    # Batch=2, Visual Patches=4, Dim=16

# Forward pass
out_tokens, self_attn_w, cross_attn_w = gated_block(sim_text, sim_visuals)

print("Output Text Tokens Shape:", out_tokens.shape)
print("Self-Attention Weights Shape:", self_attn_w.shape)
print("Cross-Attention Weights Shape:", cross_attn_w.shape)
print("Initial gate parameter value:", gated_block.gated_cross_attn.tanh_gate.item())
print("Initial multiplier factor (tanh(0)):", torch.tanh(gated_block.gated_cross_attn.tanh_gate).item())

Output Text Tokens Shape: torch.Size([2, 5, 16])
Self-Attention Weights Shape: torch.Size([2, 2, 5, 5])
Cross-Attention Weights Shape: torch.Size([2, 2, 5, 4])
Initial gate parameter value: 0.0
Initial multiplier factor (tanh(0)): 0.0


### 23. Visual Token Compression Benchmark

When scaling up image resolution (e.g. processing $224 \times 224$ images or larger), the number of patch tokens grows quickly ($32, 64, 196,$ or $576$ tokens). Passing all of these raw tokens directly to the LLM increases computational overhead quadratically.

Here, we benchmark our **Q-Former Resampler** by evaluating how it compresses various input visual token scales down to a compact bottleneck sequence size.

In [80]:
import time
import torch

# Configure our benchmark parameters
token_configurations = [32, 64, 196, 576]
bottleneck_queries = 8
embed_dim = 16
batch_size = 4

# Re-initialize a standard resampler
benchmark_resampler = QFormerResampler(num_queries=bottleneck_queries, embed_dim=embed_dim)
benchmark_resampler.eval()

print(f"=== Visual Token Compression Benchmark ===")
print(f"Target Bottleneck Output Tokens: {bottleneck_queries}\n")

for num_tokens in token_configurations:
    # Generate simulated raw visual tokens representing different patch counts
    raw_visual_tokens = torch.randn(batch_size, num_tokens, embed_dim)

    # Measure forward pass time
    t_start = time.perf_counter()
    with torch.no_grad():
        compressed_out, _ = benchmark_resampler(raw_visual_tokens)
    t_end = time.perf_counter()

    latency_ms = (t_end - t_start) * 1000
    compression_ratio = num_tokens / bottleneck_queries

    print(f"Raw Input Tokens: {num_tokens:<4} | Output Tokens: {bottleneck_queries} | Compression: {compression_ratio:.1f}x | Latency: {latency_ms:.3f} ms")

=== Visual Token Compression Benchmark ===
Target Bottleneck Output Tokens: 8

Raw Input Tokens: 32   | Output Tokens: 8 | Compression: 4.0x | Latency: 1.880 ms
Raw Input Tokens: 64   | Output Tokens: 8 | Compression: 8.0x | Latency: 0.632 ms
Raw Input Tokens: 196  | Output Tokens: 8 | Compression: 24.5x | Latency: 0.653 ms
Raw Input Tokens: 576  | Output Tokens: 8 | Compression: 72.0x | Latency: 0.982 ms


### 24. Digital Signal Processing (DSP): Waveforms, DFT, and STFT

To expand our capabilities into audio and sensory multimodal processing, we will implement core **Digital Signal Processing (DSP)** algorithms from scratch:
1. **Waveform Generation**: Synthesize composite periodic signals (sine waves) with additive high-frequency noise.
2. **Discrete Fourier Transform (DFT)**: Transform time-domain waveforms into their corresponding frequency-domain spectrum to identify constituent frequencies.
3. **Short-Time Fourier Transform (STFT)**: Compute localized Fourier Transforms using sliding window intervals (e.g., Hann windows) to construct a time-frequency spectrogram representation.

In [81]:
import numpy as np
import matplotlib.pyplot as plt

def generate_composite_waveform(sampling_rate=1000, duration=1.5):
    """Generates a composite signal containing 50Hz and 120Hz sine waves with additive noise"""
    t = np.linspace(0, duration, int(sampling_rate * duration), endpoint=False)
    # Construct 50 Hz and 120 Hz composite signal
    signal = 1.5 * np.sin(2 * np.pi * 50 * t) + 2.0 * np.sin(2 * np.pi * 120 * t)
    # Add white Gaussian noise
    noise = np.random.normal(0, 0.8, len(t))
    waveform = signal + noise
    return t, waveform

# Generate sample time domain waveform
samp_rate = 1000
time_steps, sample_waveform = generate_composite_waveform(sampling_rate=samp_rate)

In [82]:
def discrete_fourier_transform(x):
    """Computes the 1D Discrete Fourier Transform (DFT) of a sequence"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    # Exponential Fourier core matrix
    exponent = -2j * np.pi * k * n / N
    W = np.exp(exponent)
    return np.dot(W, x)

# Compute frequency spectra
dft_output = discrete_fourier_transform(sample_waveform)
# Calculate two-sided and one-sided amplitude spectrum
N = len(sample_waveform)
freqs = np.fft.fftfreq(N, d=1/samp_rate)
amplitudes = np.abs(dft_output) / N

# Keep only positive frequencies for representation
positive_indices = freqs >= 0
pos_freqs = freqs[positive_indices]
pos_amplitudes = amplitudes[positive_indices] * 2

In [83]:
def short_time_fourier_transform(x, fs, nperseg=256, noverlap=128):
    """Computes the Short-Time Fourier Transform (STFT) from raw inputs"""
    step = nperseg - noverlap
    # Hann Window implementation
    window = 0.5 * (1 - np.cos(2 * np.pi * np.arange(nperseg) / (nperseg - 1)))

    # Slice signals into overlapping windowed segments
    stft_matrix = []
    for start in range(0, len(x) - nperseg + 1, step):
        segment = x[start : start + nperseg] * window
        # Standard Fast Fourier Transform on the local segment
        segment_dft = np.fft.fft(segment)
        stft_matrix.append(segment_dft[:nperseg // 2 + 1])

    stft_matrix = np.array(stft_matrix).T

    # Map frequency and time axes
    freq_axis = np.linspace(0, fs / 2, nperseg // 2 + 1)
    time_axis = np.arange(stft_matrix.shape[1]) * (step / fs)

    return freq_axis, time_axis, stft_matrix

f_axis, t_axis, spectrogram = short_time_fourier_transform(sample_waveform, fs=samp_rate)

In [84]:
# Print quantitative metrics
print("Time-domain Signal Shape:", sample_waveform.shape)
print("DFT Coefficients Computed Shape:", dft_output.shape)
print("STFT Spectrogram Matrix Shape:", spectrogram.shape)
print(f"Detected peak spectral frequency: {pos_freqs[np.argmax(pos_amplitudes)]:.1f} Hz")

Time-domain Signal Shape: (1500,)
DFT Coefficients Computed Shape: (1500,)
STFT Spectrogram Matrix Shape: (129, 10)
Detected peak spectral frequency: 120.0 Hz


### 25. Mel Spectrogram: Mel Filter Banks and Log-Mel Features

To make our audio representations suitable for machine learning models, we convert the linear frequency scale of our STFT spectrogram into the **Mel scale**, which aligns more closely with human hearing sensitivity. This is achieved by:
1. **Mel scale conversion**: Converting frequencies between Hertz ($f$) and Mel ($m$) domains:
   $$m = 2595 \log_{10}\left(1 + \frac{f}{700}\right)$$
2. **Mel Filter Banks**: Constructing overlapping triangular filters spaced linearly on the Mel scale.
3. **Log-Mel Spectrogram**: Multiplying our STFT power spectrogram by the Mel filter bank and taking the logarithm to scale amplitude decibels logarithmically.

In [85]:
import numpy as np

def hz_to_mel(hz):
    return 2595 * np.log10(1 + hz / 700.0)

def mel_to_hz(mel):
    return 700 * (10**(mel / 2595.0) - 1)

def create_mel_filterbank(num_filters=26, nfft=256, sample_rate=1000):
    """Generates a triangular Mel filter bank matrix"""
    low_freq = 0
    high_freq = sample_rate / 2

    # Convert bounds to Mel scale and find equally spaced points
    low_mel = hz_to_mel(low_freq)
    high_mel = hz_to_mel(high_freq)
    mel_points = np.linspace(low_mel, high_mel, num_filters + 2)
    hz_points = mel_to_hz(mel_points)

    # Map Hz points to FFT bin indices
    bins = np.floor((nfft + 1) * hz_points / sample_rate).astype(int)

    # Construct triangular filters
    fbank = np.zeros((num_filters, int(nfft / 2 + 1)))
    for m in range(1, num_filters + 1):
        f_m_minus = bins[m - 1]
        f_m = bins[m]
        f_m_plus = bins[m + 1]

        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bins[m - 1]) / (bins[m] - bins[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bins[m + 1] - k) / (bins[m + 1] - bins[m])

    return fbank

In [86]:
# Calculate power spectrogram
power_spectrogram = np.abs(spectrogram) ** 2

# Create a Mel filter bank with 20 filters
mel_filters = create_mel_filterbank(num_filters=20, nfft=256, sample_rate=samp_rate)

# Apply filter banks to power spectrogram
mel_spectrogram = np.dot(mel_filters, power_spectrogram)

# Compute log-Mel features with a small epsilon offset to prevent log(0)
log_mel_spectrogram = 10 * np.log10(np.maximum(mel_spectrogram, 1e-10))

print("Mel Filter Bank Matrix Shape:", mel_filters.shape)
print("Output Log-Mel Spectrogram Shape:", log_mel_spectrogram.shape)
print("Min dB Value:", np.min(log_mel_spectrogram))
print("Max dB Value:", np.max(log_mel_spectrogram))

Mel Filter Bank Matrix Shape: (20, 129)
Output Log-Mel Spectrogram Shape: (20, 10)
Min dB Value: 18.90509584085862
Max dB Value: 42.922327929008155


### 26. Speech & Audio Encoders: Comparing Whisper and BEATs Representations

To move beyond custom digital signal processing (STFT/Log-Mel) and incorporate pre-trained high-level speech and sound semantic extractors, we integrate:
1. **Whisper (OpenAI)**: An encoder trained on weak speech-to-text supervision, optimized for phoneme density, speaker semantics, and robust noise features.
2. **BEATs (Microsoft)**: A self-supervised acoustic tokenized model designed to capture diverse non-speech acoustic patterns, environment events, and generic soundscapes.

Let's extract representations from both models for a synthesized/sample wave signal, aligning their frame rates and comparing their embedding spaces.

In [87]:
!pip install -q transformers librosa

In [88]:
import torch
import numpy as np
import librosa
from transformers import WhisperProcessor, WhisperModel, AutoProcessor, AutoModel

# 1. Synthesize a 16kHz audio waveform (standard sampling rate for Whisper and BEATs)
# Let's generate a 3-second signal containing a sliding tone (chirp) representing speech-like variation
fs = 16000
duration = 3.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)
sweep = np.sin(2 * np.pi * 300 * t + 150 * np.pi * (t ** 2)) # Dynamic acoustic signal
audio_input = sweep.astype(np.float32)

print("Generated Audio Waveform Shape:", audio_input.shape)
print("Sampling rate:", fs, "Hz")

Generated Audio Waveform Shape: (48000,)
Sampling rate: 16000 Hz


In [89]:
# 2. Extract Hidden State Representations using Whisper Encoder
# We use the 'openai/whisper-tiny' model for fast execution in Colab
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
whisper_model = WhisperModel.from_pretrained("openai/whisper-tiny")
whisper_encoder = whisper_model.encoder

# Preprocess audio (Whisper expects exactly 30 seconds, padded automatically by the processor)
inputs_whisper = whisper_processor(audio_input, sampling_rate=fs, return_tensors="pt")
input_features = inputs_whisper.input_features

with torch.no_grad():
    whisper_outputs = whisper_encoder(input_features)
    # Extracted representation from the last encoder layer block
    whisper_embeddings = whisper_outputs.last_hidden_state

print("--- Whisper Representation ---")
print("Input Mel Spectrogram Features Shape:", input_features.shape) # [Batch, Filters=80, Frames=3000]
print("Encoder Last Hidden State Shape:   ", whisper_embeddings.shape) # [Batch, Time_Steps=1500, Hidden_Dim=384]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

--- Whisper Representation ---
Input Mel Spectrogram Features Shape: torch.Size([1, 80, 3000])
Encoder Last Hidden State Shape:    torch.Size([1, 1500, 384])


In [91]:
# 3. Extract Representations using BEATs Audio Encoder
# BEATs maps generic acoustics and audio structures into tokenized feature spaces
try:
    # Attempt to load the public Microsoft BEATs checkpoint via Hugging Face
    beats_processor = AutoProcessor.from_pretrained("microsoft/beats-base")
    beats_model = AutoModel.from_pretrained("microsoft/beats-base")

    inputs_beats = beats_processor(audio_input, sampling_rate=fs, return_tensors="pt")

    with torch.no_grad():
        beats_outputs = beats_model(**inputs_beats)
        beats_embeddings = beats_outputs.last_hidden_state

    print("--- BEATs Representation ---")
    print("Input Features Processed Shape:", inputs_beats["input_values"].shape)
    print("Encoder Last Hidden State Shape:", beats_embeddings.shape)
except Exception as e:
    print("--- BEATs Load Exception caught (401 Unauthorized or Gated Repo) ---")
    print("Error details:", e)
    print("Using simulated BEATs features [Batch=1, Frames=50, Hidden_Dim=768] to preserve execution downstream.")

    # Fallback to simulated BEATs output to match target shape: [Batch, Frames, Hidden_Dim=768]
    beats_embeddings = torch.randn(1, 50, 768)

--- BEATs Load Exception caught (401 Unauthorized or Gated Repo) ---
Error details: microsoft/beats-base is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Using simulated BEATs features [Batch=1, Frames=50, Hidden_Dim=768] to preserve execution downstream.


In [92]:
from transformers import ASTFeatureExtractor, ASTModel

try:
    print("--- Loading Alternative Open Model: Audio Spectrogram Transformer (AST) ---")
    # AST is fully open, un-gated, and hosted publicly on Hugging Face
    ast_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset")
    ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset")

    # Preprocess our 16kHz audio input
    inputs_ast = ast_extractor(audio_input, sampling_rate=fs, return_tensors="pt")

    with torch.no_grad():
        ast_outputs = ast_model(**inputs_ast)
        ast_embeddings = ast_outputs.last_hidden_state

    print("\n--- AST Representation ---")
    print("Input Features Processed Shape:", inputs_ast["input_values"].shape)
    print("AST Encoder Last Hidden State Shape:", ast_embeddings.shape) # [Batch, Sequence_Len, Hidden_Dim=768]

    # Assign to our downstream beats_embeddings variable so the rest of the notebook executes seamlessly!
    beats_embeddings = ast_embeddings

except Exception as e:
    print("Failed to load AST alternative:", e)


--- Loading Alternative Open Model: Audio Spectrogram Transformer (AST) ---
Failed to load AST alternative: MIT/ast-finetuned-audioset is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


In [93]:
# 4. Compare Dimensionality and Informational Bottleneck
print("=== Representation Comparison ===")
print(f"Whisper Sequence Dimensions: {list(whisper_embeddings.shape)}")
print(f"BEATs Sequence Dimensions:   {list(beats_embeddings.shape)}")

# Standardize temporal steps via average pooling to evaluate correlation across shared duration
whisper_pool = torch.nn.functional.adaptive_avg_pool1d(whisper_embeddings.transpose(1, 2), output_size=50).squeeze(0).T
beats_pool = torch.nn.functional.adaptive_avg_pool1d(beats_embeddings.transpose(1, 2), output_size=50).squeeze(0).T

print("\nTemporal-Aligned Embeddings Shape (50 pooled steps):")
print("Whisper Downsampled Shape:", whisper_pool.shape)
print("BEATs Downsampled Shape:  ", beats_pool.shape)

# Compute internal cosine similarity dynamics along timeline to see if both models transition together
whisper_timeline_sim = torch.matmul(whisper_pool, whisper_pool.t()) / (
    torch.norm(whisper_pool, dim=-1, keepdim=True) * torch.norm(whisper_pool, dim=-1).t()
)
beats_timeline_sim = torch.matmul(beats_pool, beats_pool.t()) / (
    torch.norm(beats_pool, dim=-1, keepdim=True) * torch.norm(beats_pool, dim=-1).t()
)

print("\nSimilarity Matrix verification:")
print("Whisper Cross-Time Similarity Matrix Shape:", whisper_timeline_sim.shape)
print("BEATs Cross-Time Similarity Matrix Shape:  ", beats_timeline_sim.shape)

=== Representation Comparison ===
Whisper Sequence Dimensions: [1, 1500, 384]
BEATs Sequence Dimensions:   [1, 50, 768]

Temporal-Aligned Embeddings Shape (50 pooled steps):
Whisper Downsampled Shape: torch.Size([50, 384])
BEATs Downsampled Shape:   torch.Size([50, 768])

Similarity Matrix verification:
Whisper Cross-Time Similarity Matrix Shape: torch.Size([50, 50])
BEATs Cross-Time Similarity Matrix Shape:   torch.Size([50, 50])


### 27. Audio Adapter: Temporal Pooling and Projection into GPT

When connecting speech encoders (like Whisper, which yields 1500 frames for 30s) or audio encoders (like BEATs) to a decoder-only LLM, we face a sequence length mismatch. Passing raw acoustic frames directly into the decoder consumes the model's context window.

To align these modalities, we implement an **Audio Adapter** that:
1. Downsamples the sequence along the temporal dimension using **Adaptive Average Pooling**.
2. Projects the downsampled features into the LLM's joint text representation space using a **Projection Layer** (Linear/MLP projection).

In [94]:
import torch
import torch.nn as nn

class AudioAdapter(nn.Module):
    def __init__(self, audio_dim, text_dim, target_seq_len=8):
        super().__init__()
        self.target_seq_len = target_seq_len
        # Linear projection to map from audio representation dimensions to the LLM's text embedding dimensions
        self.projection = nn.Linear(audio_dim, text_dim)
        self.ln = nn.LayerNorm(text_dim)

    def forward(self, audio_features):
        # audio_features shape: [Batch, Seq_Len, Audio_Dim]
        # Transpose to pool along the temporal sequence dimension
        x = audio_features.transpose(1, 2) # [Batch, Audio_Dim, Seq_Len]

        # 1. Temporal pooling to scale down sequence length
        pooled = torch.nn.functional.adaptive_avg_pool1d(x, output_size=self.target_seq_len)
        pooled = pooled.transpose(1, 2) # [Batch, Target_Seq_Len, Audio_Dim]

        # 2. Map features to LLM embedding dimension
        projected = self.projection(pooled)
        return self.ln(projected)

In [95]:
# Demonstration: Adapting both Whisper and BEATs representations to our MiniGPT
text_dim = 16 # MiniGPT embed_dim

# Initialize Audio Adapters
whisper_adapter = AudioAdapter(audio_dim=384, text_dim=text_dim, target_seq_len=8)
beats_adapter = AudioAdapter(audio_dim=768, text_dim=text_dim, target_seq_len=8)

# Run projections
projected_whisper = whisper_adapter(whisper_embeddings)
projected_beats = beats_adapter(beats_embeddings)

print("=== Adapted Audio Representations ===")
print("Original Whisper shape:", whisper_embeddings.shape)
print("Adapted Whisper shape: ", projected_whisper.shape)
print("Original BEATs shape:  ", beats_embeddings.shape)
print("Adapted BEATs shape:   ", projected_beats.shape)

# Concatenate adapted audio prefixes with text tokens to simulate multimodal input
prompt_text_ids = torch.tensor([[tokenizer.vocab['learning'], tokenizer.vocab['embeddings']]])
prompt_embeddings = gpt_model.token_embeddings(prompt_text_ids) # [1, 2, 16]

multimodal_input = torch.cat([projected_whisper, prompt_embeddings], dim=1)
print("\nUnified Multimodal Input Sequence Shape (8 Audio + 2 Text tokens):", multimodal_input.shape)

=== Adapted Audio Representations ===
Original Whisper shape: torch.Size([1, 1500, 384])
Adapted Whisper shape:  torch.Size([1, 8, 16])
Original BEATs shape:   torch.Size([1, 50, 768])
Adapted BEATs shape:    torch.Size([1, 8, 16])

Unified Multimodal Input Sequence Shape (8 Audio + 2 Text tokens): torch.Size([1, 10, 16])


### 28. Speech-aware GPT: End-to-End Training on Spoken Instructions

Now, we build **Speech-aware GPT** by coupling our frozen speech/sound encoders with our language model.

We will set up a training pipeline where:
1. Spoken audio commands are encoded (via Whisper or BEATs) and compressed into a fixed sequence of **8 audio prefix tokens** using our `AudioAdapter`.
2. The decoder-only `MiniGPT` takes these 8 prefix tokens concatenated with text instruction tokens to perform next-token autoregressive generation.
3. We train the network on a joint speech-instruction task (answering spoken prompts) using standard Cross-Entropy Loss.

In [96]:
import torch
import torch.nn as nn
import torch.optim as optim

class SpeechAwareGPT(nn.Module):
    def __init__(self, base_gpt, audio_adapter):
        super().__init__()
        self.gpt = base_gpt
        self.audio_adapter = audio_adapter

        # Freeze the main GPT layers to perform resource-efficient audio prefix tuning
        for p in self.gpt.parameters():
            p.requires_grad = False

        # The Audio Adapter remains fully trainable
        for p in self.audio_adapter.parameters():
            p.requires_grad = True

    def forward(self, audio_features, target_token_ids):
        # 1. Adapt and compress audio representation: [Batch, 8, Text_Dim]
        audio_prefix = self.audio_adapter(audio_features)

        # 2. Get target text embeddings: [Batch, Seq_Len, Text_Dim]
        text_embeddings = self.gpt.token_embeddings(target_token_ids)

        # 3. Concatenate adapted audio prefix tokens with text tokens
        combined_inputs = torch.cat([audio_prefix, text_embeddings], dim=1) # [Batch, 8 + Seq_Len, Text_Dim]

        # 4. Generate positional coordinates and run forward pass
        batch_size, total_len, _ = combined_inputs.shape
        positions = torch.arange(0, total_len, device=target_token_ids.device).unsqueeze(0)
        x = combined_inputs + self.gpt.position_embeddings(positions)

        for block in self.gpt.blocks:
            x, _ = block(x)

        logits = self.gpt.lm_head(self.gpt.ln_f(x))
        return logits

In [97]:
# 1. Prepare speech training pairs (Spoken Instruction Audio -> Text Response)
# We map simulated acoustic arrays representing words to target texts
spoken_instructions = [
    ("audio_query_learning", "learning embeddings makes neural networks powerful"),
    ("audio_query_deep", "deep learning is fascinating")
]

# Tokenize the training texts
all_instruction_texts = [item[1] for item in spoken_instructions]
tokenizer.fit(all_instruction_texts)

# Prepare input tensors and targets
max_seq_len = 10
X_audio = torch.randn(2, 1500, 384) # Simulate Whisper representations for 2 audio items
Y_text = []

for _, text in spoken_instructions:
    encoded = tokenizer.encode(text, max_length=max_seq_len)
    Y_text.append(encoded)

Y_text = torch.tensor(Y_text, dtype=torch.long)

# Shift inputs/targets by 1 for next-token prediction
input_ids = Y_text[:, :-1]
target_ids = Y_text[:, 1:]

In [98]:
# 2. Instantiate and pretrain our Speech-aware GPT
base_gpt_audio = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=24
)

whisper_adapter_trainable = AudioAdapter(audio_dim=384, text_dim=16, target_seq_len=8)
speech_gpt = SpeechAwareGPT(base_gpt_audio, whisper_adapter_trainable)

# Configure Optimization targeting ONLY the audio adapter parameters
optimizer = optim.Adam(speech_gpt.audio_adapter.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 3. Training Loop
speech_gpt.train()
print("--- Training Speech-aware GPT Model ---")
for epoch in range(100):
    optimizer.zero_grad()

    # Forward pass: outputs are [Batch, 8 + Seq_Len - 1, Vocab_Size]
    logits = speech_gpt(X_audio, input_ids)

    # Slice out only the text generation portion (ignoring the first 8 audio prefix steps)
    text_logits = logits[:, 8:]

    loss = criterion(text_logits.reshape(-1, len(tokenizer.vocab)), target_ids.reshape(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:03d} / 100 | Speech Instruction Loss: {loss.item():.4f}")

--- Training Speech-aware GPT Model ---
Epoch 025 / 100 | Speech Instruction Loss: 3.1672
Epoch 050 / 100 | Speech Instruction Loss: 3.1543
Epoch 075 / 100 | Speech Instruction Loss: 3.1509
Epoch 100 / 100 | Speech Instruction Loss: 3.1495


In [99]:
# 4. Test Autoregressive Generation Conditioned on Audio
speech_gpt.eval()
with torch.no_grad():
    print("\n--- Verification of Conditioned Spoken Output ---")
    # Use first audio feature representation
    sample_audio = X_audio[0:1]
    prompt_ids = torch.tensor([[tokenizer.vocab['learning']]], dtype=torch.long)

    # Run a single decoding forward pass
    logits = speech_gpt(sample_audio, prompt_ids)
    predicted_token_id = torch.argmax(logits[:, -1, :], dim=-1).item()

    print("Input Spoken Instruction target text:", spoken_instructions[0][1])
    print("Prompt input token:                  ", tokenizer.decode([tokenizer.vocab['learning']]))
    print("Predicted next token from audio context:", tokenizer.decode([predicted_token_id]))


--- Verification of Conditioned Spoken Output ---
Input Spoken Instruction target text: learning embeddings makes neural networks powerful
Prompt input token:                   learning
Predicted next token from audio context: describe


### 29. General Audio Understanding: Sound Captioning and Audio Question Answering (AQA)

To build a versatile audio assistant, we expand our model's instruction tuning to handle **General Audio Understanding**. This involves training the network across two different paradigms:
1. **Sound Captioning**: Outputting a descriptive caption based solely on the background acoustic events.
2. **Audio Question Answering (AQA)**: Interpreting a question about the audio and producing a precise answer.

We use our pre-trained `BEATs` representation space (as it is optimized for acoustic events and environment classification) and project it into `MiniGPT` using an optimized `AudioAdapter`.

In [100]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Create a multi-task dataset
audio_understanding_data = [
    # (Acoustic Context, Instruction Prompt, Target Answer)
    ("dog_bark_audio", "describe the sound", "dog barking loudly"),
    ("dog_bark_audio", "is there an animal", "yes a dog"),
    ("siren_noise_audio", "describe the sound", "emergency siren wailing"),
    ("siren_noise_audio", "is it high pitch", "yes high pitch")
]

# Fit tokenizer on all new textual elements
all_multitask_texts = [item[1] for item in audio_understanding_data] + [item[2] for item in audio_understanding_data]
tokenizer.fit(all_multitask_texts)

print("Expanded Multitask Vocab Size:", len(tokenizer.vocab))
print("Tokenizer vocabulary sample:", list(tokenizer.vocab.keys())[:15])

Expanded Multitask Vocab Size: 45
Tokenizer vocabulary sample: ['<PAD>', '<UNK>', 'and', 'building', 'deep', 'embeddings', 'fascinating', 'is', 'it', 'language', 'learning', 'love', 'make', 'natural', 'networks']


In [101]:
# 2. Prepare multi-task training arrays with aligned sequences
max_seq_len = 16

# Simulated BEATs feature representations for our distinct audio sources
# Let's mock 2 base audio source embeddings [Length=100, Dim=768] matching 'dog' and 'siren'
dog_feats = torch.randn(100, 768) + 1.2
siren_feats = torch.randn(100, 768) - 1.2

audio_source_map = {
    "dog_bark_audio": dog_feats,
    "siren_noise_audio": siren_feats
}

X_audio_batch = []
Y_input_ids = []
Y_target_ids = []

for audio_key, prompt, answer in audio_understanding_data:
    X_audio_batch.append(audio_source_map[audio_key])

    # Full sequence format: "<prompt> <answer>"
    full_sequence = f"{prompt} {answer}"
    encoded = tokenizer.encode(full_sequence, max_length=max_seq_len)

    # Slice out input and target offset arrays for next-token autoregressive training
    Y_input_ids.append(encoded[:-1])
    Y_target_ids.append(encoded[1:])

X_audio_batch = torch.stack(X_audio_batch) # [Batch=4, Seq=100, Dim=768]
Y_input_ids = torch.tensor(Y_input_ids, dtype=torch.long) # [Batch=4, Seq_Len-1]
Y_target_ids = torch.tensor(Y_target_ids, dtype=torch.long) # [Batch=4, Seq_Len-1]

print("Acoustic input batch shape:        ", X_audio_batch.shape)
print("Text prompt-response input shape:   ", Y_input_ids.shape)

Acoustic input batch shape:         torch.Size([4, 100, 768])
Text prompt-response input shape:    torch.Size([4, 15])


In [102]:
# 3. Instantiate and train our Multitask Audio Assistant
base_gpt_multitask = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=32
)

# Define the BEATs adapter mapping 768-dim features to 16-dim GPT spaces
beats_adapter_multitask = AudioAdapter(audio_dim=768, text_dim=16, target_seq_len=8)
multitask_audio_gpt = SpeechAwareGPT(base_gpt_multitask, beats_adapter_multitask)

# Configure optimizer for prefix adaptation
optimizer = optim.Adam(multitask_audio_gpt.audio_adapter.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Train loop
multitask_audio_gpt.train()
print("--- Training Multitask Audio Assistant ---")
for epoch in range(120):
    optimizer.zero_grad()

    # Forward pass: Outputs [Batch, 8 + Seq_Len - 1, Vocab_Size]
    logits = multitask_audio_gpt(X_audio_batch, Y_input_ids)

    # Exclude the 8 leading audio prefix tokens to calculate text-loss
    text_logits = logits[:, 8:]

    loss = criterion(text_logits.reshape(-1, len(tokenizer.vocab)), Y_target_ids.reshape(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:03d} / 120 | Multitask Audio Loss: {loss.item():.4f}")

--- Training Multitask Audio Assistant ---
Epoch 030 / 120 | Multitask Audio Loss: 3.7895
Epoch 060 / 120 | Multitask Audio Loss: 3.7718
Epoch 090 / 120 | Multitask Audio Loss: 3.7610
Epoch 120 / 120 | Multitask Audio Loss: 3.7534


In [103]:
# 4. Evaluate General Audio Understanding Tasks
multitask_audio_gpt.eval()
with torch.no_grad():
    print("\n--- Verification of General Audio Understanding Predictions ---")
    for i, (audio_key, prompt, target_ans) in enumerate(audio_understanding_data):
        single_audio = X_audio_batch[i:i+1]
        prompt_tokens = torch.tensor([tokenizer.encode(prompt, max_length=4)], dtype=torch.long)

        # Retrieve generated tokens past the audio sequence bottleneck prefix
        logits = multitask_audio_gpt(single_audio, prompt_tokens)
        text_logits = logits[:, 8:]
        predicted_ids = torch.argmax(text_logits, dim=-1)[0].tolist()

        print(f"\n[Audio context: {audio_key} | Query: '{prompt}']")
        print("Target Answer:     ", target_ans)
        print("Model Output:      ", tokenizer.decode(predicted_ids))


--- Verification of General Audio Understanding Predictions ---

[Audio context: dog_bark_audio | Query: 'describe the sound']
Target Answer:      dog barking loudly
Model Output:       what makes deep

[Audio context: dog_bark_audio | Query: 'is there an animal']
Target Answer:      yes a dog
Model Output:       what sat yes networks

[Audio context: siren_noise_audio | Query: 'describe the sound']
Target Answer:      emergency siren wailing
Model Output:       what sat makes deep

[Audio context: siren_noise_audio | Query: 'is it high pitch']
Target Answer:      yes high pitch
Model Output:       what sat yes siren


### 30. Sensory Compression: Vector Quantization (VQ) Implementation from Scratch

To discretize continuous high-dimensional audio or visual feature spaces into a compact set of codes (reminiscent of models like VQ-VAE or SoundStream), we implement a **Vector Quantizer** layer.

Key details:
1. **Trainable Codebook**: A set of $K$ continuous latent embedding vectors of dimensionality $D$.
2. **Nearest-Neighbor Quantization**: Map each continuous input vector $z_e(x)$ to its closest codebook entry $e_i$.
3. **Straight-Through Estimator (STE)**: Copy gradients directly from the decoder inputs back to the encoder outputs ($z_q \approx z_e$).
4. **Loss Formulation**: Include Commitment Loss (ensuring continuous encoders stay close to codebook vectors) and Codebook Loss (pushing codebook vectors toward encoder outputs).

In [108]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=16, embedding_dim=8, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        # Trainable codebook
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)
        self.codebook.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, inputs):
        # inputs shape: [Batch, Seq_Len, Dim] (e.g. continuous acoustic/visual states)
        B, S, D = inputs.shape
        assert D == self.embedding_dim, "Input feature dimension must match embedding dimension"

        # Flatten inputs
        flat_inputs = inputs.view(-1, D)

        # 1. Compute pairwise Euclidean distances between inputs and codebook vectors
        # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a · b
        distances = (
            torch.sum(flat_inputs**2, dim=1, keepdim=True)
            + torch.sum(self.codebook.weight**2, dim=1)
            - 2 * torch.matmul(flat_inputs, self.codebook.weight.t())
        )

        # 2. Find nearest indices
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encoding_indices = encoding_indices.view(B, S)

        # 3. Retrieve quantized vectors
        quantized = self.codebook(encoding_indices) # [Batch, Seq_Len, Dim]

        # 4. Calculate VQ losses
        # Codebook loss: push codebook embeddings closer to encoder outputs
        loss_codebook = F.mse_loss(quantized, inputs.detach())
        # Commitment loss: discourage encoder outputs from fluctuating too wildly
        loss_commitment = F.mse_loss(quantized.detach(), inputs)
        vq_loss = loss_codebook + self.commitment_cost * loss_commitment

        # 5. Straight-Through Estimator (STE)
        # Trick to copy gradients backward: 'quantized' gets gradient of inputs directly
        quantized = inputs + (quantized - inputs).detach()

        # 6. Calculate perplexity to track codebook usage distribution
        # Convert indices to one-hot distributions
        flat_indices = encoding_indices.view(-1)
        encodings = F.one_hot(flat_indices, num_classes=self.num_embeddings).float()
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        return quantized, encoding_indices, vq_loss, perplexity

In [109]:
# 1. Re-initialize Vector Quantizer (using the corrected class definition)
num_codes = 8
code_dim = 16
vq_layer = VectorQuantizer(num_embeddings=num_codes, embedding_dim=code_dim, commitment_cost=0.25)

# 2. Create sample continuous sensory representations
# Batch size of 2, 4 tokens, 16 dimensions each
sample_sensory_features = torch.randn(2, 4, code_dim)

# 3. Quantize input representations
quantized_vectors, indices, vq_loss, perplexity = vq_layer(sample_sensory_features)

print("=== Vector Quantization Metrics ===")
print("Continuous Input shape:    ", sample_sensory_features.shape)
print("Quantized Output shape:    ", quantized_vectors.shape)
print("Discrete Codebook Indices: \n", indices)
print(f"Total VQ Loss:              {vq_loss.item():.5f}")
print(f"Active Codebook Perplexity: {perplexity.item():.3f} (out of {num_codes} possible entries)")

=== Vector Quantization Metrics ===
Continuous Input shape:     torch.Size([2, 4, 16])
Quantized Output shape:     torch.Size([2, 4, 16])
Discrete Codebook Indices: 
 tensor([[6, 0, 1, 1],
        [3, 3, 5, 3]])
Total VQ Loss:              1.36395
Active Codebook Perplexity: 4.456 (out of 8 possible entries)


### 31. Neural Audio Codec: Waveform Quantization and Reconstruction

To see our Vector Quantization (VQ) system in action with raw sensory data, we assemble a miniature **Neural Audio Codec** (analogous to SoundStream or EnCodec).

Our system consists of:
1. **Encoder**: A series of convolutional layers that downsample raw 1D waveforms into continuous sequential features.
2. **VQ Bottleneck**: Discretizes these continuous features into codebook indices.
3. **Decoder**: Transposed convolutional layers that upsample the quantized features back to the original time-domain waveform resolution.

In [110]:
import torch
import torch.nn as nn

class AudioEncoder(nn.Module):
    def __init__(self, out_channels=16):
        super().__init__()
        # Downsamples raw 1D audio waveform by a factor of 4
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=15, stride=1, padding=7),
            nn.ELU(),
            nn.Conv1d(8, 16, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
            nn.Conv1d(16, out_channels, kernel_size=4, stride=2, padding=1)
        )

    def forward(self, x):
        # Input shape: [Batch, Channels=1, Waveform_Samples]
        return self.encoder(x)

class AudioDecoder(nn.Module):
    def __init__(self, in_channels=16):
        super().__init__()
        # Upsamples features back to original temporal resolution
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(in_channels, 16, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
            nn.ConvTranspose1d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
            nn.Conv1d(8, 1, kernel_size=15, stride=1, padding=7)
        )

    def forward(self, x):
        # Input shape: [Batch, Latent_Dim, Seq_Len]
        return self.decoder(x)

In [111]:
class NeuralAudioCodec(nn.Module):
    def __init__(self, num_embeddings=16, embedding_dim=16):
        super().__init__()
        self.encoder = AudioEncoder(out_channels=embedding_dim)
        self.vq = VectorQuantizer(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.decoder = AudioDecoder(in_channels=embedding_dim)

    def forward(self, waveform):
        # 1. Encode waveform to continuous representations
        continuous_latents = self.encoder(waveform) # [B, Dim, Seq_Len]

        # Permute to fit VQ input expectations [B, Seq_Len, Dim]
        continuous_latents_permuted = continuous_latents.transpose(1, 2)

        # 2. Vector Quantize continuous features into discrete codes
        quantized, indices, vq_loss, perplexity = self.vq(continuous_latents_permuted)

        # Permute back to decoder input shape [B, Dim, Seq_Len]
        quantized_permuted = quantized.transpose(1, 2)

        # 3. Decode back to time-domain waveform
        reconstructed_waveform = self.decoder(quantized_permuted)

        return reconstructed_waveform, indices, vq_loss, perplexity

In [112]:
# 1. Instantiate the Codec
codec = NeuralAudioCodec(num_embeddings=16, embedding_dim=16)

# 2. Create a simulated raw audio waveform (1D sequence with a channel dimension)
# Shape: [Batch=1, Channels=1, Samples=128]
simulated_audio = torch.sin(torch.linspace(0, 4 * 3.1415, 128)).unsqueeze(0).unsqueeze(0)

# 3. Pass through the Neural Audio Codec pipeline
reconstructed_audio, discrete_codes, loss_vq, usage_perplexity = codec(simulated_audio)

print("=== Neural Audio Codec Execution Output ===")
print("Input Audio Waveform Shape:        ", list(simulated_audio.shape))
print("Compressed Discrete Audio Codes Shape:", list(discrete_codes.shape))
print("Reconstructed Audio Waveform Shape: ", list(reconstructed_audio.shape))
print("\n--- Extracted Discrete Codes (Acoustic Index Tokens) ---")
print(discrete_codes[0].tolist())
print(f"\nInitial VQ loss:                    {loss_vq.item():.5f}")
print(f"Active Codebook Perplexity:         {usage_perplexity.item():.4f}")

=== Neural Audio Codec Execution Output ===
Input Audio Waveform Shape:         [1, 1, 128]
Compressed Discrete Audio Codes Shape: [1, 32]
Reconstructed Audio Waveform Shape:  [1, 1, 128]

--- Extracted Discrete Codes (Acoustic Index Tokens) ---
[3, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 9, 9, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 15]

Initial VQ loss:                    0.01596
Active Codebook Perplexity:         3.6565


### 32. Autoregressive Audio Generation Network (Predicting Codec Tokens)

Instead of generating raw, highly-redundant time-domain float values directly, modern generative audio models (like AudioLM, MusicGen, or Bark) first convert audio into discrete acoustic tokens using a neural codec, and then train an autoregressive language-like decoder to model and predict these discrete codebook indices over time.

Here, we build a **Causal Audio Generator**:
1. **Acoustic GPT Decoder**: A causal transformer designed to predict the sequence distribution of discrete audio tokens from our codebook.
2. **Waveform Decoder Synthesis**: Decode predicted tokens back into fully reconstructed raw waveforms via the `NeuralAudioCodec` decoder.

### 37. Learned Latent Memory Workspace (Replacing Prefix Fusion)

Rather than appending raw sensory adapters' outputs directly to our instruction sequence, we use a **Learned Latent Workspace** layer. In this setup:
1. A fixed set of learnable **Workspace Latent Memory** tokens acts as a central bottleneck.
2. Cross-attention blocks allow these latents to query and compress the incoming visual and acoustic feature spaces successively.
3. The compressed multi-sensory memory sequence is then injected into the language model. This protects the context length from exploding when handling diverse modalities.

In [113]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LatentMemoryWorkspace(nn.Module):
    def __init__(self, num_memory_tokens=4, embed_dim=16, num_heads=2):
        super().__init__()
        # Fixed length learned latent parameters acting as the global memory bottleneck
        self.memory_tokens = nn.Parameter(torch.randn(1, num_memory_tokens, embed_dim))

        # Cross-Attention: Latent Queries attend to visual/audio features
        self.vision_cross_attn = CrossAttentionBottleneck(embed_dim=embed_dim, num_heads=num_heads)
        self.audio_cross_attn = CrossAttentionBottleneck(embed_dim=embed_dim, num_heads=num_heads)

        self.norm_mem = nn.LayerNorm(embed_dim)
        self.norm_vis = nn.LayerNorm(embed_dim)
        self.norm_aud = nn.LayerNorm(embed_dim)

    def forward(self, visual_features, audio_features):
        # visual_features: [Batch, Vis_Len, Embed_Dim]
        # audio_features: [Batch, Aud_Len, Embed_Dim]
        batch_size = visual_features.shape[0]

        # Expand learnable memory tokens to batch size
        mem = self.memory_tokens.expand(batch_size, -1, -1) # [Batch, Num_Memory_Tokens, Embed_Dim]

        # 1. Update memory workspace by querying the Visual features
        vis_updates, _ = self.vision_cross_attn(self.norm_mem(mem), self.norm_vis(visual_features))
        mem = mem + vis_updates

        # 2. Update memory workspace by querying the Audio features
        aud_updates, _ = self.audio_cross_attn(self.norm_mem(mem), self.norm_aud(audio_features))
        mem = mem + aud_updates

        return mem

# Verification of the Latent Memory Fusion Workspace
workspace = LatentMemoryWorkspace(num_memory_tokens=4, embed_dim=16, num_heads=2)

# Simulated input features mapped to unified embed_dim=16
sim_vis = torch.randn(2, 9, 16)
sim_aud = torch.randn(2, 8, 16)

fused_workspace_memory = workspace(sim_vis, sim_aud)

print("=== Latent Memory Workspace Analysis ===")
print("Input Visual Tokens Shape:", list(sim_vis.shape))
print("Input Audio Tokens Shape: ", list(sim_aud.shape))
print("Fused Latent Memory Shape:", list(fused_workspace_memory.shape))

=== Latent Memory Workspace Analysis ===
Input Visual Tokens Shape: [2, 9, 16]
Input Audio Tokens Shape:  [2, 8, 16]
Fused Latent Memory Shape: [2, 4, 16]


### 38. Unified Cognitive Integration & Benchmarking Suite

We will now execute an end-to-end integration benchmark that connects our components:
1. **Sensory Bottleneck Compression**: Measure the dimensionality compaction of the `LatentMemoryWorkspace` versus raw prefix-fusion methods.
2. **Modular Expert Specialization**: Verify that specialized tokens route dynamically to their corresponding modal experts.
3. **Recurrent vs. Attention Bottleneck Analysis**: Compare sequence processing latencies and memory requirements between our models.

In [114]:
import torch
import torch.nn as nn
import torch.optim as optim

class AcousticTokenGenerator(nn.Module):
    def __init__(self, num_codes=16, embed_dim=16, num_heads=2, hidden_features=32, num_layers=2):
        super().__init__()
        # Standard discrete token embeddings representing our VQ codebook indices
        self.embeddings = nn.Embedding(num_codes, embed_dim)
        self.pos_embeddings = nn.Embedding(64, embed_dim) # Max seq length of 64

        self.blocks = nn.ModuleList([
            GPTTransformerBlock(embed_dim, num_heads, hidden_features)
            for _ in range(num_layers)
        ])

        self.ln = RMSNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_codes)

    def forward(self, token_ids):
        # token_ids: [Batch, Seq_Len]
        B, S = token_ids.shape
        positions = torch.arange(0, S, device=token_ids.device).unsqueeze(0)

        # Add token and positional coordinate features
        x = self.embeddings(token_ids) + self.pos_embeddings(positions)

        # Forward pass through causal attention blocks
        for block in self.blocks:
            x, _ = block(x)

        logits = self.head(self.ln(x))
        return logits

In [115]:
# 1. Prepare discrete training target sequences extracted from our audio codec
codec.eval()
with torch.no_grad():
    # Encode our simulated audio to retrieve the true ground-truth discrete codes
    _, ground_truth_codes, _, _ = codec(simulated_audio)
    # ground_truth_codes shape: [1, Seq_Len=32]

print("Target Discrete Acoustic Codes:\n", ground_truth_codes[0].tolist())

# Configure training data for shifting targets (next token prediction)
X_audio_tokens = ground_truth_codes[:, :-1]
Y_audio_targets = ground_truth_codes[:, 1:]

Target Discrete Acoustic Codes:
 [3, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 9, 9, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 15]


In [116]:
# 2. Train the Acoustic Token Generator
audio_generator = AcousticTokenGenerator(num_codes=16, embed_dim=16, num_heads=2, hidden_features=32, num_layers=2)
optimizer = optim.Adam(audio_generator.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

audio_generator.train()
print("--- Training Acoustic Token Generator ---")
for epoch in range(100):
    optimizer.zero_grad()
    logits = audio_generator(X_audio_tokens)
    loss = criterion(logits.view(-1, 16), Y_audio_targets.view(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:03d} / 100 | Next-Acoustic-Token CrossEntropy Loss: {loss.item():.4f}")

--- Training Acoustic Token Generator ---
Epoch 025 / 100 | Next-Acoustic-Token CrossEntropy Loss: 0.0437
Epoch 050 / 100 | Next-Acoustic-Token CrossEntropy Loss: 0.0045
Epoch 075 / 100 | Next-Acoustic-Token CrossEntropy Loss: 0.0025
Epoch 100 / 100 | Next-Acoustic-Token CrossEntropy Loss: 0.0019


In [117]:
# 3. Perform Autoregressive Audio Token Generation & Waveform Synthesis
audio_generator.eval()
with torch.no_grad():
    # Start with a seed token representing the initial acoustic event
    generated_tokens = [ground_truth_codes[0, 0].item()]

    for _ in range(31):  # Autoregressively generate next 31 tokens
        input_tensor = torch.tensor([generated_tokens], dtype=torch.long)
        logits = audio_generator(input_tensor)
        next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
        generated_tokens.append(next_token)

    print("\nGenerated Discrete Code Sequence:  ", generated_tokens)
    print("Target Ground Truth Code Sequence: ", ground_truth_codes[0].tolist())

    # 4. Synthesize raw waveform back from generated codes using Neural Audio Codec Decoder
    # Retrieve embedding vectors corresponding to the generated code indices
    generated_indices_tensor = torch.tensor([generated_tokens], dtype=torch.long)
    quantized_latents = codec.vq.codebook(generated_indices_tensor)  # [Batch=1, Seq=32, Dim=16]

    # Permute to fit Decoder convolutional footprint [Batch, Dim, Seq]
    quantized_latents_permuted = quantized_latents.transpose(1, 2)
    synthesized_waveform = codec.decoder(quantized_latents_permuted)

    print("\n--- Sound Synthesis Verification ---")
    print("Synthesized Output Waveform Shape: ", list(synthesized_waveform.shape))


Generated Discrete Code Sequence:   [3, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 9, 9, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 15]
Target Ground Truth Code Sequence:  [3, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 9, 9, 3, 3, 3, 3, 10, 10, 10, 15, 15, 15, 15, 15, 15, 9, 15]

--- Sound Synthesis Verification ---
Synthesized Output Waveform Shape:  [1, 1, 128]


### 33. OmniGPT: Unified Vision, Audio, and Text Fusion Network

To build a truly multi-sensory intelligence system, we now integrate all three core modalities (**Vision**, **Audio**, and **Text**) into a single unified sequence processor: **OmniGPT**.

#### Fusion Strategy:
1. **Visual Projection**: Compress raw images into localized sequence representations and project them to the joint space via `MLPVisionAdapter`.
2. **Acoustic Projection**: Extract and pool high-dimensional environmental sound hidden states via `AudioAdapter`.
3. **Unified Multimodal Input Sequence**: Concatenate the continuous sensory tokens (Vision + Audio) as structured multi-modal prefixes to our text instructions, enabling unified self-attention to process all modalities concurrently.

In [118]:
import torch
import torch.nn as nn
import torch.optim as optim

class OmniGPT(nn.Module):
    def __init__(self, base_gpt, vision_adapter, audio_adapter):
        super().__init__()
        self.gpt = base_gpt
        self.vision_adapter = vision_adapter
        self.audio_adapter = audio_adapter

        # Freeze base language model to preserve prior knowledge
        for p in self.gpt.parameters():
            p.requires_grad = False

        # Adapters remain fully trainable to map vision and audio features to the text space
        for p in self.vision_adapter.parameters():
            p.requires_grad = True
        for p in self.audio_adapter.parameters():
            p.requires_grad = True

    def forward(self, image_features, audio_features, text_ids):
        # 1. Project Visual features: [Batch, Vis_Seq, Text_Dim]
        vis_projected = self.vision_adapter(image_features)

        # 2. Project and Pool Audio features: [Batch, Audio_Seq, Text_Dim]
        audio_projected = self.audio_adapter(audio_features)

        # 3. Get text token embeddings: [Batch, Text_Seq, Text_Dim]
        text_embeddings = self.gpt.token_embeddings(text_ids)

        # 4. Concatenate all three modalities: [Batch, Vis_Seq + Audio_Seq + Text_Seq, Text_Dim]
        multimodal_sequence = torch.cat([vis_projected, audio_projected, text_embeddings], dim=1)

        # Generate coordinates for combined sequence length
        batch_size, total_seq_len, _ = multimodal_sequence.shape
        positions = torch.arange(0, total_seq_len, device=text_ids.device).unsqueeze(0)

        # Add spatial/positional embeddings
        x = multimodal_sequence + self.gpt.position_embeddings(positions)

        # Run through stacked Transformer blocks
        for block in self.gpt.blocks:
            x, _ = block(x)

        logits = self.gpt.lm_head(self.gpt.ln_f(x))
        return logits

In [119]:
# Initialize joint instruction-tuning dataset containing images, sounds, queries, and target answers
omni_multimodal_dataset = [
    # (Image Source, Audio Source, Text Question, Target Answer)
    ("cat_img", "dog_bark_audio", "what is in the scene", "cat hears dog barking")
]

# Fit tokenizer on our new multi-modal instruction-tuning texts
all_omni_texts = [item[2] for item in omni_multimodal_dataset] + [item[3] for item in omni_multimodal_dataset]
tokenizer.fit(all_omni_texts)

print("Omni-Modal Vocab Size:", len(tokenizer.vocab))
print("Omni-Modal Vocab Map:", tokenizer.vocab)

Omni-Modal Vocab Size: 48
Omni-Modal Vocab Map: {'<PAD>': 0, '<UNK>': 1, 'and': 2, 'building': 3, 'deep': 4, 'embeddings': 5, 'fascinating': 6, 'is': 7, 'it': 8, 'language': 9, 'learning': 10, 'love': 11, 'make': 12, 'natural': 13, 'networks': 14, 'neural': 15, 'powerful': 16, 'processing': 17, 'we': 18, 'cat': 19, 'count': 20, 'describe': 21, 'image': 22, 'objects': 23, 'on': 24, 'sat': 25, 'this': 26, 'two': 27, 'what': 28, 'makes': 29, 'a': 30, 'an': 31, 'animal': 32, 'barking': 33, 'dog': 34, 'emergency': 35, 'high': 36, 'loudly': 37, 'pitch': 38, 'siren': 39, 'sound': 40, 'the': 41, 'there': 42, 'wailing': 43, 'yes': 44, 'hears': 45, 'in': 46, 'scene': 47}


In [120]:
# 1. Prepare continuous sensory representations
sim_image_input = torch.randn(1, 3, 12, 12)
sim_image_patches = patch_proc.patchify_conv(sim_image_input) # Extract 9 visual patches [1, 9, 16]
sim_audio_features = torch.randn(1, 100, 768) # Simulate BEATs acoustic representations

# Tokenize text inputs with shifting targets
max_seq_len = 12
full_prompt_text = f"{omni_multimodal_dataset[0][2]} {omni_multimodal_dataset[0][3]}"
encoded_prompt = tokenizer.encode(full_prompt_text, max_length=max_seq_len)

X_text_tokens = torch.tensor([encoded_prompt[:-1]], dtype=torch.long)
Y_text_targets = torch.tensor([encoded_prompt[1:]], dtype=torch.long)

In [122]:
# 2. Instantiate and train our OmniGPT Model
base_gpt_omni = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=48
)

# Create adapters to project our diverse visual and audio shapes into 16-dimensional GPT tokens
vision_adapter_omni = MLPVisionAdapter(vision_dim=16, text_dim=16, hidden_dim=32)
audio_adapter_omni = AudioAdapter(audio_dim=768, text_dim=16, target_seq_len=8)

omni_model = OmniGPT(base_gpt_omni, vision_adapter_omni, audio_adapter_omni)

# Optimize both adapter networks simultaneously
trainable_params = list(omni_model.vision_adapter.parameters()) + list(omni_model.audio_adapter.parameters())
optimizer = optim.Adam(trainable_params, lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Detach simulated features from any previous computational graph history to prevent backward errors
train_image_patches = sim_image_patches.detach()
train_audio_features = sim_audio_features.detach()

omni_model.train()
print("--- Training Unified OmniGPT (Vision + Audio + Text) ---")
for epoch in range(120):
    optimizer.zero_grad()

    # Forward pass: 9 visual tokens + 8 audio tokens + 11 text input tokens
    logits = omni_model(train_image_patches, train_audio_features, X_text_tokens)

    # Isolate text prediction logits: exclude the first 17 sensory prefix steps (9 vision + 8 audio)
    text_logits = logits[:, 17:]

    loss = criterion(text_logits.reshape(-1, len(tokenizer.vocab)), Y_text_targets.reshape(-1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:03d} / 120 | Omni-Modal Loss: {loss.item():.4f}")

--- Training Unified OmniGPT (Vision + Audio + Text) ---
Epoch 030 / 120 | Omni-Modal Loss: 4.2668
Epoch 060 / 120 | Omni-Modal Loss: 4.2604
Epoch 090 / 120 | Omni-Modal Loss: 4.2590
Epoch 120 / 120 | Omni-Modal Loss: 4.2586


In [123]:
# 3. Evaluate Omni-Modal Tri-Fusion Performance
omni_model.eval()
with torch.no_grad():
    print("--- Evaluating Unified Omni-Modal Inference ---")
    question = omni_multimodal_dataset[0][2]
    prompt_tokens = torch.tensor([tokenizer.encode(question, max_length=6)], dtype=torch.long)

    # Generate prediction
    logits = omni_model(sim_image_patches, sim_audio_features, prompt_tokens)
    text_logits = logits[:, 17:]
    predicted_ids = torch.argmax(text_logits, dim=-1)[0].tolist()

    print("\nVisual Scene Input:       Image of a cat")
    print("Acoustic Context Input:    Sound of dog barking")
    print("Text Question Prompt:      ", question)
    print("Target Joint Answer:        cat hears dog barking")
    print("OmniGPT Fused Prediction: ", tokenizer.decode(predicted_ids))

--- Evaluating Unified Omni-Modal Inference ---

Visual Scene Input:       Image of a cat
Acoustic Context Input:    Sound of dog barking
Text Question Prompt:       what is in the scene
Target Joint Answer:        cat hears dog barking
OmniGPT Fused Prediction:  barking dog barking sat dog there


### 34. Video Understanding: Frame Sampling and Temporal Compression

Unlike static images, video streams introduce a massive temporal dimension $[Batch, Frames, Channels, Height, Width]$ with significant cross-frame redundancy. Passing every pixel patch from every frame directly into our language decoder quickly saturates the model's sequence limit.

To handle video natively, we build:
1. **Spatio-Temporal Frame Sampler**: Extracts a fixed number of uniformly spaced frames from raw video streams.
2. **3D Spatio-Temporal Compression Block**: Uses a 3D Convolution layer followed by temporal average-pooling to compress high-rate frame representations into a compact sequence of prefix visual tokens suitable for our language decoder.

In [124]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VideoFrameSampler:
    def __init__(self, num_target_frames=4):
        self.num_target_frames = num_target_frames

    def sample_uniformly(self, video_tensor):
        # video_tensor shape: [Batch, Frames, Channels, Height, Width]
        B, F_len, C, H, W = video_tensor.shape

        if F_len <= self.num_target_frames:
            # Zero-pad frames if video sequence is shorter than target budget
            padding = torch.zeros(B, self.num_target_frames - F_len, C, H, W, device=video_tensor.device)
            return torch.cat([video_tensor, padding], dim=1)

        # Uniform index selection
        indices = torch.linspace(0, F_len - 1, steps=self.num_target_frames, dtype=torch.long)
        sampled_frames = video_tensor[:, indices]
        return sampled_frames

In [125]:
class SpatioTemporalVideoCompressor(nn.Module):
    def __init__(self, in_channels=3, embed_dim=16, patch_size=4, target_temporal_tokens=4):
        super().__init__()
        self.target_temporal_tokens = target_temporal_tokens

        # 3D Convolutional projector: extracts features across Space and Time simultaneously
        # Inputs: [Batch, Channels, Frames, Height, Width]
        self.conv3d = nn.Conv3d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=(3, patch_size, patch_size),
            stride=(1, patch_size, patch_size),
            padding=(1, 0, 0)
        )
        self.ln = nn.LayerNorm(embed_dim)

    def forward(self, sampled_video):
        # sampled_video shape: [Batch, Frames, Channels, Height, Width]
        # Rearrange to 3D Conv format: [Batch, Channels, Frames, Height, Width]
        x = sampled_video.permute(0, 2, 1, 3, 4)

        # Run 3D convolution
        features = self.conv3d(x) # [Batch, Embed_Dim, Out_Frames, Out_H, Out_W]

        # Flatten spatial dimensions and pool temporally to self.target_temporal_tokens
        B, D, T, H, W = features.shape
        flat_spatial = features.view(B, D, T, H * W).permute(0, 2, 3, 1) # [Batch, T, H*W, Embed_Dim]

        # Flatten temporal and spatial positions into a sequential structure
        flat_sequence = flat_spatial.reshape(B, T * H * W, D)

        # Pool temporally down to our target token length to prevent sequence explosion
        pooled_sequence = F.adaptive_avg_pool1d(flat_sequence.transpose(1, 2), self.target_temporal_tokens)
        pooled_sequence = pooled_sequence.transpose(1, 2)

        return self.ln(pooled_sequence)

In [126]:
# 1. Simulate video processing pipeline
video_sampler = VideoFrameSampler(num_target_frames=4)
video_compressor = SpatioTemporalVideoCompressor(in_channels=3, embed_dim=16, patch_size=4, target_temporal_tokens=6)

# Mock a raw video input sequence of 10 frames, 3 channels, size 12x12
simulated_raw_video = torch.randn(2, 10, 3, 12, 12)

# Step A: Sample frames uniformly
sampled_frames = video_sampler.sample_uniformly(simulated_raw_video)

# Step B: Compress Spatio-Temporal representation
video_tokens = video_compressor(sampled_frames)

print("=== Video Sampling and Compression Verification ===")
print("Original Video input tensor:        ", list(simulated_raw_video.shape))
print("Uniformly sampled video subset:    ", list(sampled_frames.shape))
print("Temporal-Compressed Output Tokens: ", list(video_tokens.shape))

=== Video Sampling and Compression Verification ===
Original Video input tensor:         [2, 10, 3, 12, 12]
Uniformly sampled video subset:     [2, 4, 3, 12, 12]
Temporal-Compressed Output Tokens:  [2, 6, 16]


### 35. Streaming Multimodal Inference & Memory Profiling

To analyze how continuous multi-sensory inputs affect downstream deployment performance, we will benchmark:
1. **Inference Latency**: Time required to process sequential streams of mixed-modality tokens.
2. **Context Consumption**: Cumulative sequence length utilization across single-modality vs. multi-sensory inputs.
3. **KV Cache Memory Footprint**: Tracking the growth of Key and Value tensors in our causal attention layers over long context windows.

In [127]:
import time
import torch
import pandas as pd

def profile_inference(model, text_tokens, num_latents=0, run_name="Text Only"):
    model.eval()
    device = next(model.parameters()).device

    # Convert tokens to tensor
    idx = torch.tensor([text_tokens], dtype=torch.long, device=device)
    batch_size = idx.shape[0]

    # Simulate sensory adaptation tokens if any
    # e.g., mapping video/audio/image features as a prefix
    prefix_tokens = num_latents

    t_start = time.perf_counter()
    with torch.no_grad():
        # 1. Prefill phase (Generates initial keys and values)
        logits, kv_caches = model(idx)

        # Track next predicted token
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        # 2. Sequential incremental decoding (simulate next 5 steps)
        start_pos = idx.shape[1] + prefix_tokens
        for _ in range(5):
            logits, kv_caches = model(next_token, kv_caches=kv_caches, start_pos=start_pos)
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            start_pos += 1

    t_end = time.perf_counter()
    latency_ms = (t_end - t_start) * 1000

    # Calculate KV Cache memory consumption in bytes
    # kv_caches shape: [Layers][K or V][B, H, Seq, Dim]
    total_elements = 0
    if kv_caches is not None:
        for layer_cache in kv_caches:
            # key and value tensors are elements of the tuple
            total_elements += layer_cache[0].nelement()
            total_elements += layer_cache[1].nelement()

    kv_bytes = total_elements * 4  # float32 uses 4 bytes
    total_context_tokens = start_pos

    return {
        "Configuration": run_name,
        "Prefill + 5-step Latency (ms)": round(latency_ms, 2),
        "Total Active Context (Tokens)": total_context_tokens,
        "KV Cache Memory (KB)": round(kv_bytes / 1024, 3)
    }

# Initialize Cached OmniGPT config for simulation
profiling_gpt = CachedMiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=16,
    num_heads=2,
    hidden_features=32,
    num_layers=2,
    max_seq_len=64
)

# Sample prompt inputs
prompt_text = "describe the scene"
prompt_tokens = tokenizer.encode(prompt_text, max_length=5)[:-1]

# Profile different streaming modalities
results = []

# 1. Text-Only Baseline
results.append(profile_inference(profiling_gpt, prompt_tokens, num_latents=0, run_name="Text Only Baseline"))

# 2. Audio-Text (8 Audio Tokens Prefix)
results.append(profile_inference(profiling_gpt, prompt_tokens, num_latents=8, run_name="Audio-Text Stream (8 Aud)"))

# 3. Vision-Text (9 Vision Tokens Prefix)
results.append(profile_inference(profiling_gpt, prompt_tokens, num_latents=9, run_name="Vision-Text Stream (9 Vis)"))

# 4. Omni-Modal (17 sensory prefix tokens: 9 Vision + 8 Audio)
results.append(profile_inference(profiling_gpt, prompt_tokens, num_latents=17, run_name="Omni-Modal (9 Vis + 8 Aud)"))

# Display summary statistics
df_benchmark = pd.DataFrame(results)
display(df_benchmark)

,Configuration,Prefill + 5-step Latency (ms),Total Active Context (Tokens),KV Cache Memory (KB)
0,Text Only Baseline,5.62,9,2.25
1,Audio-Text Stream (8 Aud),5.02,17,2.25
2,Vision-Text Stream (9 Vis),5.26,18,2.25
3,Omni-Modal (9 Vis + 8 Aud),5.11,26,2.25


### 36. Multimodal Expert Routing (Dynamic Perception Gating)

Instead of treating all sensory modalities uniformly inside our joint attention blocks, we implement **Multimodal Expert Routing**. This module dynamically routes sequence tokens to specialized perception experts (e.g., a Vision Expert, an Audio Expert, or a Text/Language Expert) based on gate routing coefficients computed directly from the latent activations of the input sequence.

In [128]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SensoryExpert(nn.Module):
    """Specialized perception expert block for a specific sensory domain."""
    def __init__(self, embed_dim, hidden_dim, modality_name="Generic"):
        super().__init__()
        self.modality_name = modality_name
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

class MultimodalExpertRouter(nn.Module):
    def __init__(self, embed_dim, hidden_dim=32, num_experts=3):
        super().__init__()
        self.num_experts = num_experts

        # Specialized perception experts: 0=Vision, 1=Audio, 2=Language
        self.experts = nn.ModuleList([
            SensoryExpert(embed_dim, hidden_dim, modality_name="Vision_Expert"),
            SensoryExpert(embed_dim, hidden_dim, modality_name="Audio_Expert"),
            SensoryExpert(embed_dim, hidden_dim, modality_name="Language_Expert")
        ])

        # Gating network to map sequence activations to expert indices
        self.gate = nn.Linear(embed_dim, num_experts, bias=False)

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Embed_Dim]
        orig_shape = x.shape
        x_flat = x.view(-1, orig_shape[-1])  # Flatten to [Tokens, Embed_Dim]

        # 1. Compute routing logits and softmax probability distributions
        gate_logits = self.gate(x_flat) # [Tokens, Num_Experts]
        routing_weights = F.softmax(gate_logits, dim=-1)

        # 2. Select top expert for each token (hard routing with scaling)
        top_weights, top_indices = torch.topk(routing_weights, k=1, dim=-1)

        output_flat = torch.zeros_like(x_flat)

        # 3. Route tokens dynamically to selected experts
        for i, expert in enumerate(self.experts):
            # Mask showing which tokens are routed to expert i
            token_mask = (top_indices == i).squeeze(-1)

            if token_mask.any():
                expert_inputs = x_flat[token_mask]
                expert_outputs = expert(expert_inputs)

                # Scale the outputs by the routing confidence weight
                weights = top_weights[token_mask]
                output_flat[token_mask] = weights * expert_outputs

        return output_flat.view(orig_shape), routing_weights

In [129]:
# Initialize the Multimodal Router
multimodal_router = MultimodalExpertRouter(embed_dim=16, hidden_dim=32, num_experts=3)
multimodal_router.eval()

# Simulate a multimodal token sequence representing a unified perception pass:
# - 4 Vision adapter tokens
# - 4 Audio adapter tokens
# - 4 Text tokens
vision_tokens = torch.randn(1, 4, 16) + 1.5  # High-amplitude positive bias
audio_tokens = torch.randn(1, 4, 16) - 1.5   # Negative bias
text_tokens = torch.randn(1, 4, 16) * 0.1     # Low variance baseline

joint_sequence = torch.cat([vision_tokens, audio_tokens, text_tokens], dim=1) # [1, 12, 16]

# Run the forward routing pass
routed_output, weights = multimodal_router(joint_sequence)

print("=== Multimodal Expert Routing Analysis ===")
print("Input Unified Sequence Shape: ", list(joint_sequence.shape))
print("Routed Output Sequence Shape:", list(routed_output.shape))

# Show routing allocations
expert_names = ["Vision Expert", "Audio Expert", "Language Expert"]
for idx in range(12):
    mod_label = "Vision" if idx < 4 else "Audio" if idx < 8 else "Text"
    selected_expert = torch.argmax(weights[idx]).item()
    print(f"Token {idx:02d} | Modality: {mod_label:<6} -> Assigned to: {expert_names[selected_expert]:<15} | Routing Probs: {weights[idx].detach().numpy().round(3)}")

=== Multimodal Expert Routing Analysis ===
Input Unified Sequence Shape:  [1, 12, 16]
Routed Output Sequence Shape: [1, 12, 16]
Token 00 | Modality: Vision -> Assigned to: Vision Expert   | Routing Probs: [0.612 0.226 0.162]
Token 01 | Modality: Vision -> Assigned to: Audio Expert    | Routing Probs: [0.244 0.603 0.153]
Token 02 | Modality: Vision -> Assigned to: Audio Expert    | Routing Probs: [0.114 0.808 0.078]
Token 03 | Modality: Vision -> Assigned to: Audio Expert    | Routing Probs: [0.367 0.465 0.168]
Token 04 | Modality: Audio  -> Assigned to: Language Expert | Routing Probs: [0.452 0.093 0.455]
Token 05 | Modality: Audio  -> Assigned to: Language Expert | Routing Probs: [0.416 0.069 0.514]
Token 06 | Modality: Audio  -> Assigned to: Language Expert | Routing Probs: [0.26  0.052 0.688]
Token 07 | Modality: Audio  -> Assigned to: Language Expert | Routing Probs: [0.308 0.223 0.468]
Token 08 | Modality: Text   -> Assigned to: Audio Expert    | Routing Probs: [0.332 0.341 0.327]